# Analisis Kualitas Udara AQMS — Anggota 2 (Revisi v4)

**Tujuan.** Mengolah data mentah AQMS (27 stasiun, 1 Jan 2024 – 14 Okt 2026) menjadi tabel siap pakai untuk dashboard Looker Studio, dengan satu aturan pengecualian anomali yang konsisten untuk semua keluaran.

**Input.** Satu file zip `Dataset DA Gelar Rasa 2026.zip` (27 file `*_AQMS.csv` + `sites.csv`). Tidak butuh file lain.

**Output.** Satu file `hasil_anggota2_v4.zip` berisi `Anggota_2_data_v4_siap_looker.xlsx` (10 sheet) dan folder `csv_audit/` (CSV bukti pendukung); terunduh otomatis di sel terakhir.

### Alur kerja
| Tahap | Bagian | Hasil |
|---|---|---|
| A | 1. Persiapan | zip diambil, data diekstrak, konfigurasi + metadata stasiun |
| B | 2. Pembersihan data | tabel jam-an bersih (659.664 baris = 27 stasiun × 24.432 jam) |
| C | 3. Agregasi harian | tabel harian per stasiun, daftar stasiun layak PM2.5 |
| D | 4. Deteksi anomali | `DUGAAN_STASIUN`, kolom `pm25_dipakai`, fungsi `tanpa_semua_anomali` |
| E | 5. Pola jam × hari | sheet heatmap untuk Looker |
| F | 6–8. Analisis | `weekday_weekend`, `episode_pm25`, `korelasi_cuaca` |
| G | 9. Ekspor | workbook Excel + ringkasan temuan + zip hasil |

### Istilah penting
- **`pm25_dipakai`** — PM2.5 harian yang sudah lolos kelayakan dan bebas anomali; dipakai di semua grafik (bukan `pm25` mentah).
- **Anomali KTG** — stasiun KTG-01 dan KTG-02 pada/setelah 1 Sep 2026 (88 hari-stasiun, belum terverifikasi).
- **Dugaan anomali 8–14 Okt 2026** — stasiun yang naik/turun mendadak lalu naik linear (*ramp*) atau datar; terdeteksi otomatis (Bagian 4).
- **`tanpa_semua_anomali`** — satu fungsi yang membuang kedua jenis anomali di atas; dipakai untuk **semua polutan**, bukan hanya PM2.5.

### Cara menjalankan
Runtime → **Jalankan semua**, lalu upload zip dataset saat dialog muncul. Bagian 8 (bootstrap korelasi) paling lama, sekitar 10–15 menit.

## Parameter yang bisa diubah
| Parameter | Nilai | Letak | Arti |
|---|---|---|---|
| `MIN_JAM_VALID` | 18 | Bagian 1 | hari dianggap valid bila ≥ 18 jam valid |
| `FLATLINE_JAM` | 6 | Bagian 1 | ≥ 6 jam nilai identik berturut-turut dianggap sensor macet |
| `AMBANG_PM25` | 55,4 µg/m³ | Bagian 1 | batas atas kategori "sedang" ISPU; dasar definisi episode |
| `MIN_EPISODE_HARI` | 3 | Bagian 1 | episode = ≥ 3 hari berturut-turut di atas ambang |
| `MIN_COV_PM` | 0,60 | Bagian 3 | stasiun layak PM2.5 bila ≥ 60% hari punya data valid |
| `ANOMALI_MULAI` | 2026-09-01 | Bagian 1 | awal periode anomali KTG |
| `DUGAAN_MULAI/AKHIR` | 8–14 Okt 2026 | Bagian 4 | jendela dugaan anomali |
| `BOOT_B` | 1000 | Bagian 8 | jumlah resample bootstrap (100–200 untuk uji cepat) |

# 1. Persiapan
## 1.1 Ambil zip dataset
**Tujuan:** mendapatkan file zip di `/content`.
**Proses:** mencari zip di `/content`; bila tidak ada, membuka dialog upload.
**Hasil:** variabel `ZIP_PATH` berisi lokasi zip.

In [ ]:
# ==== 1.1 AMBIL ZIP DATASET ====
import os, glob, zipfile

ZIP_PATH = None
for z in glob.glob("/content/*.zip"):
    if zipfile.is_zipfile(z) and "hasil_anggota2" not in os.path.basename(z):
        ZIP_PATH = z; break

if ZIP_PATH is None:
    from google.colab import files
    print("Pilih file zip dataset (Dataset DA Gelar Rasa 2026.zip) dari komputermu ...")
    for nama in files.upload():
        if nama.lower().endswith(".zip"):
            ZIP_PATH = "/content/" + nama; break

assert ZIP_PATH and zipfile.is_zipfile(ZIP_PATH), "Zip dataset belum ada / belum lengkap."
print(f"OK. Zip siap: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")

OK. Zip siap: /content/Dataset DA Gelar Rasa 2026.zip (18.5 MB)


## 1.2 Ekstrak data dan siapkan folder
**Tujuan:** membuka zip dan memastikan 27 file stasiun lengkap.
**Proses:** ekstrak ke `/content/data_mentah`, hitung file `*_AQMS.csv`.
**Hasil:** `DATA_DIR` (data mentah) dan `OUT_DIR` (folder keluaran CSV audit). Berhenti dengan pesan error bila file ≠ 27.

In [ ]:
# ==== 1.2 EKSTRAK DATA ====
DATA_DIR = "/content/data_mentah"
OUT_DIR = "/content/output"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

if len(glob.glob(f"{DATA_DIR}/**/*_AQMS.csv", recursive=True)) != 27:
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

files_found = glob.glob(f"{DATA_DIR}/**/*_AQMS.csv", recursive=True)
print(f"File AQMS ditemukan: {len(files_found)} (harus 27)")
assert len(files_found) == 27, "Dataset AQMS mentah harus berisi tepat 27 file."

File AQMS ditemukan: 27 (harus 27)


## 1.3 Konfigurasi analisis
**Tujuan:** menetapkan semua parameter di satu tempat.
**Isi:** periode data, daftar polutan, ambang kualitas data (flatline, jam valid, ambang ISPU), periode anomali KTG, dan daftar stasiun yang sensornya berunit ppm (BJB-01, MDN-01, PSW-01) yang harus dikonversi ke µg/m³.
**Catatan:** stasiun koreksi waktu (BTM-01, JMB-01, MLG-01, PKN-01, STG-01) adalah stasiun yang timestamp awalnya jatuh 23:4x–23:5x hari sebelumnya.

In [ ]:
# ==== 1.3 KONFIGURASI ====
# ============================================================
import os, glob
import numpy as np
import pandas as pd

# ---------- KONFIGURASI WAKTU / KONVERSI ----------
STASIUN_KOREKSI = ["BTM-01", "JMB-01", "MLG-01", "PKN-01", "STG-01"]   # timestamp mulai 23:4x-23:5x hari sebelumnya (terdeteksi dari data)
ZONA_DIDUKUNG_DOKUMENTASI = True    # booklet case study menyebut timestamp_start dan timestamp_end dalam WIB
KONFIRMASI_ZONA_NAIVE = "WIB"     # didukung keterangan WIB pada Booklet Case Study DA Gelar Rasa 2026 (bagian Detail Data Udara)
VOL_MOLAR_L = 24.45               # L/mol; 25 C & 1 atm. (0 C & 1 atm = 22.414)
# --------------------------------------------------

PERIODE = ("2024-01-01", "2026-10-14")           # sesuai metadata_dates.csv
POL = ["pm25", "pm10", "so2", "co", "o3", "no2"]
POL_LABEL = {"pm25": "PM2.5", "pm10": "PM10", "so2": "SO2", "co": "CO", "o3": "O3", "no2": "NO2"}
CUACA = ["temp", "rh", "ws", "wd"]
KEEP_QC = "VALID"
FLATLINE_JAM = 6                  # >= 6 jam nilai identik berturut-turut dianggap sensor macet
MIN_JAM_VALID = 18                # hari valid jika >= 18 jam valid
MIN_EPISODE_HARI = 3
AMBANG_PM25 = 55.4                # ug/m3 24 jam (batas atas kategori "sedang" ISPU)
HARI = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
MW = {"so2": 64.06, "co": 28.01, "o3": 48.00, "no2": 46.01}   # g/mol

ANOMALI_MULAI = "2026-09-01"
ANOMALI_STASIUN = ["KTG-01", "KTG-02"]
PPM_STATIONS = ["BJB-01", "MDN-01", "PSW-01"]    # kolom *_ppm di file mentah (sites.csv: unit = ppm)
PM_EXCLUDE = ["JMB-02"]                          # coverage PM sangat rendah (cell 2 menambah PPM_STATIONS)

## 1.4 Metadata stasiun dan fungsi bantu
**Tujuan:** memetakan tiap stasiun ke kawasannya dan menyiapkan fungsi yang dipakai di seluruh notebook.
**Proses:** membaca `sites.csv`; fungsi `_kawasan` mengelompokkan stasiun menjadi 5 kawasan (gambut/karhutla, industri/pelabuhan, background bersih, pesisir, perkotaan/lalu lintas).
**Fungsi:** `tanpa_anomali` (aturan lama, hanya KTG), `kategori_ispu_pm25`, `pembulatan`.
**Hasil:** `META` (27 stasiun) dan ringkasan jumlah stasiun per kawasan.

In [ ]:
# ==== 1.4 METADATA STASIUN + FUNGSI BANTU ====
def _kawasan(site_type, karakter):
    t = f"{site_type} {karakter}".lower()
    if "peatland" in t or "gambut" in t:           return "gambut/karhutla"
    if "industr" in t:                             return "industri/pelabuhan"
    if "clean" in t or "baseline" in t or "pristine" in t: return "background bersih"
    if "coastal" in t:                             return "pesisir"
    return "perkotaan/lalu lintas"

_sites_path = glob.glob(os.path.join(DATA_DIR, "**", "sites.csv"), recursive=True)[0]
_sites = pd.read_csv(_sites_path)
META = {r.location_id.replace("LOC-", ""): dict(provinsi=r.provinsi, kota=r.kota_kabupaten,
                                               kawasan=_kawasan(r.site_type, r.karakteristik_wilayah),
                                               site_type=r.site_type)
        for r in _sites.itertuples()}
assert len(META) == 27, f"META harus 27 stasiun, dapat {len(META)}"

# ---------- fungsi bantu ----------
def tanpa_anomali(df):
    """buang baris stasiun anomali pada/setelah ANOMALI_MULAI (df harus punya kolom station, date)"""
    m = df["station"].isin(ANOMALI_STASIUN) & (df["date"] >= pd.Timestamp(ANOMALI_MULAI))
    return df[~m].copy()

def kategori_ispu_pm25(x):
    if pd.isna(x): return np.nan
    return ("baik" if x <= 15.5 else "sedang" if x <= 55.4 else "tidak sehat" if x <= 150.4
            else "sangat tidak sehat" if x <= 250.4 else "berbahaya")

def pembulatan(df, nd=2):
    out = df.copy()
    num = out.select_dtypes("number").columns
    out[num] = out[num].round(nd)
    return out

print("Kawasan per stasiun:"); print(pd.Series({k: v["kawasan"] for k, v in META.items()}).value_counts())

Kawasan per stasiun:
gambut/karhutla          9
perkotaan/lalu lintas    7
industri/pelabuhan       4
background bersih        4
pesisir                  3
Name: count, dtype: int64


# 2. Pembersihan data (jam-an)
## 2.1 Fungsi pembacaan dan konversi
**Tujuan:** mendefinisikan cara membaca satu file stasiun.
**Proses per file:**
1. mengubah nama kolom ke nama baku (`pm25`, `so2`, `temp`, …);
2. nilai sentinel CO = 999 dijadikan kosong;
3. gas stasiun ppm dikonversi ke µg/m³ (rumus: ppm × berat molekul × 1000 / 24,45);
4. baris dengan `qc_flag` bukan VALID dikosongkan;
5. waktu dihitung dari titik tengah interval, lalu pengukuran dalam jam yang sama dirata-ratakan (khusus PLB-01 yang punya pengukuran 15 menit + 1 jam; arah angin dirata-ratakan secara sirkular).
**Hasil:** fungsi `baca_stasiun`.

In [ ]:
# ------------------------------------------------------------
# ==== 2.1 FUNGSI PEMBACAAN + KONVERSI ====
# ------------------------------------------------------------
COL_START, COL_END, COL_QC = "timestamp_start", "timestamp_end", "qc_flag"
SENTINEL_CO = 999
EXPECTED_ROWS = 659_664
STRICT_VALIDASI = True
KOLOM_QC_KOSONG = list(dict.fromkeys(POL + CUACA + ["wd"]))
AUDIT_BACA = []
AUDIT_TIMESTAMP_INVALID = []
RENAME_COL = {f"{g}_{u}": g for g in ("pm25", "pm10", "so2", "co", "o3", "no2") for u in ("ug_m3", "ppm")}
RENAME_COL.update({"temperature_c": "temp", "humidity_pct": "rh", "wind_speed_ms": "ws", "wind_direction_deg": "wd"})
LOG_KONVERSI = []

# Konversi ppm -> ug/m3 hanya untuk parameter gas di stasiun yang sumbernya ppm.
KONVERSI_AKTIF = True
GAS_WAJIB = [c for c in POL if c not in ("pm25", "pm10")]
if KONVERSI_AKTIF:
    assert VOL_MOLAR_L, "isi VOL_MOLAR_L sesuai asumsi temperatur dan tekanan"
    assert set(GAS_WAJIB) <= set(MW), f"MW belum mencakup gas: {sorted(set(GAS_WAJIB) - set(MW))}"
assert len(STASIUN_KOREKSI) == 5 and set(STASIUN_KOREKSI) <= set(META)
assert KONFIRMASI_ZONA_NAIVE in ("WIB", "UTC")


def konversi_gas(df):
    st = df["station"].iloc[0]
    if st not in PPM_STATIONS:
        return df
    for gas, mw in MW.items():
        if gas in df.columns:
            sebelum = df[gas].median()
            df[gas] = df[gas] * mw * 1000.0 / VOL_MOLAR_L
            LOG_KONVERSI.append(dict(station=st, gas=gas, unit_sebelum="ppm", unit_sesudah="ug/m3",
                                     MW=mw, vol_molar_L=VOL_MOLAR_L,
                                     median_sebelum=sebelum, median_sesudah=df[gas].median()))
    return df


ZONA_TARGET = "Asia/Jakarta"

def _ke_naive(s):
    s = pd.to_datetime(s, errors="coerce")
    if getattr(s.dt, "tz", None) is not None:
        return s.dt.tz_convert(ZONA_TARGET).dt.tz_localize(None)
    if KONFIRMASI_ZONA_NAIVE == "UTC":
        return s.dt.tz_localize("UTC").dt.tz_convert(ZONA_TARGET).dt.tz_localize(None)
    return s


def _mean_wd_deg(s):
    """Circular mean arah angin derajat; NaN jika tak ada observasi valid."""
    x = pd.to_numeric(s, errors="coerce").dropna()
    if x.empty:
        return np.nan
    rad = np.deg2rad(x.to_numpy(dtype=float))
    ss, cc = np.sin(rad).mean(), np.cos(rad).mean()
    if abs(ss) < 1e-12 and abs(cc) < 1e-12:
        return np.nan
    return float(np.rad2deg(np.arctan2(ss, cc)) % 360)


def baca_stasiun(path):
    df = pd.read_csv(path)
    if "station" not in df.columns:
        df["station"] = os.path.basename(path).split("_")[0]
    station = str(df["station"].iloc[0])
    n_baris_mentah = len(df)

    t0, t1 = _ke_naive(df[COL_START]), _ke_naive(df[COL_END])
    duration_min = (t1 - t0).dt.total_seconds() / 60.0
    time_mid = (t0 + (t1 - t0) / 2).dt.floor("h")
    time_start_floor = t0.dt.floor("h")
    timestamp_valid = t0.notna() & t1.notna() & duration_min.gt(0) & time_mid.notna()

    df = df.rename(columns=RENAME_COL)
    for c_ in KOLOM_QC_KOSONG:
        if c_ in df.columns:
            df[c_] = pd.to_numeric(df[c_], errors="coerce")
    kurang_ = [c for c in KOLOM_QC_KOSONG if c not in df.columns]
    assert not kurang_, f"kolom belum tersedia setelah rename: {kurang_}"

    df["is_sentinel_co"] = df["co"].eq(SENTINEL_CO)
    df.loc[df["is_sentinel_co"], "co"] = np.nan
    if KONVERSI_AKTIF:
        df = konversi_gas(df)

    df["_qc_flag"] = df[COL_QC].astype(str).str.strip().replace({"nan": "MISSING_FLAG"})
    df["_qc_flag_valid"] = df["_qc_flag"].eq(KEEP_QC)
    df["qc_ok"] = df["_qc_flag_valid"] & timestamp_valid
    df["time"] = time_mid.where(timestamp_valid)
    df["time_lama"] = time_start_floor
    df["duration_min"] = duration_min

    # Audit row-level: interval invalid dan timestamp di luar periode analisis.
    in_window = time_mid.ge(pd.Timestamp(PERIODE[0])) & time_mid.lt(pd.Timestamp(PERIODE[1]) + pd.Timedelta(days=1))
    bad = (~timestamp_valid) | (~in_window)
    if bad.any():
        reason = np.where(~timestamp_valid,
                    np.where(t0.isna() | t1.isna(), "timestamp_parse_invalid", "timestamp_end_not_after_start"),
                    "timestamp_outside_expected_period")
        AUDIT_TIMESTAMP_INVALID.extend(pd.DataFrame({
            "station": station,
            "timestamp_start": df.loc[bad, COL_START].astype(str).to_numpy(),
            "timestamp_end": df.loc[bad, COL_END].astype(str).to_numpy(),
            "duration_min": duration_min.loc[bad].to_numpy(),
            "qc_flag": df.loc[bad, COL_QC].astype(str).to_numpy(),
            "reason": reason[bad.to_numpy()]
        }).to_dict("records"))

    # QC buruk tetap dihitung di audit sumber, tetapi nilai analitik dikosongkan.
    df.loc[~df["qc_ok"], KOLOM_QC_KOSONG] = np.nan
    valid_time = df.loc[timestamp_valid].copy()
    valid_time["_wd_rad"] = np.deg2rad(pd.to_numeric(valid_time["wd"], errors="coerce"))
    valid_time["_wd_sin"] = np.sin(valid_time["_wd_rad"])
    valid_time["_wd_cos"] = np.cos(valid_time["_wd_rad"])

    # GroupBy mean/count vektor: menghindari Python loop per slot jam.
    g = valid_time.groupby("time", sort=True)
    scalar_cols = [c for c in KOLOM_QC_KOSONG if c != "wd"]
    hourly = g[scalar_cols].mean()
    wd_sin = g["_wd_sin"].mean()
    wd_cos = g["_wd_cos"].mean()
    hourly["wd"] = np.rad2deg(np.arctan2(wd_sin, wd_cos)) % 360
    hourly.loc[(wd_sin.abs() < 1e-12) & (wd_cos.abs() < 1e-12), "wd"] = np.nan
    hourly["station"] = station
    hourly["n_baris_sumber"] = g.size()
    hourly["n_qc_valid_sumber"] = g["qc_ok"].sum()
    hourly["n_qc_tidak_valid_sumber"] = g.size() - g["_qc_flag_valid"].sum()
    hourly["n_sentinel_co_sumber"] = g["is_sentinel_co"].sum()
    hourly["qc_ok"] = hourly["n_qc_valid_sumber"].gt(0)
    hourly["is_sentinel_co"] = hourly["n_sentinel_co_sumber"].gt(0)
    for c_ in KOLOM_QC_KOSONG:
        hourly[f"n_valid_{c_}_sumber"] = g[c_].count()
    hourly = hourly.reset_index()

    group_sizes = g.size()
    n_durasi_bukan_1jam = int((duration_min.notna() & (duration_min.round(4) != 60.0)).sum())
    n_geser = int((time_mid[timestamp_valid] != time_start_floor[timestamp_valid]).sum())
    AUDIT_BACA.append(dict(
        station=station,
        tz_mentah="naive" if getattr(t0.dt, "tz", None) is None else str(t0.dt.tz),
        contoh_timestamp_start=str(df[COL_START].iloc[0]) if len(df) else "",
        n_baris_mentah=n_baris_mentah,
        n_slot_jam_terbentuk=int(len(group_sizes)),
        n_jam_dengan_multi_pengukuran=int((group_sizes > 1).sum()),
        n_pengukuran_tambahan_di_jam=int((group_sizes - 1).clip(lower=0).sum()),
        n_durasi_bukan_1jam=n_durasi_bukan_1jam,
        n_geser_jam_vs_floor_start=n_geser,
        n_timestamp_interval_tidak_valid=int((~timestamp_valid).sum()),
        n_timestamp_di_luar_periode=int((~in_window).sum()),
        n_qc_tidak_valid=int((~df["_qc_flag_valid"]).sum()),
        n_qc_valid=int(df["_qc_flag_valid"].sum()),
        n_sentinel_co=int(df["is_sentinel_co"].sum())
    ))
    return hourly

## 2.2 Baca 27 stasiun dan audit awal
**Tujuan:** membaca semua file dan mendokumentasikan kondisi data mentah.
**Hasil (tabel yang dicetak):** audit timestamp per stasiun (jumlah baris, slot jam, QC tidak valid, sentinel CO) dan log konversi ppm → µg/m³ (median sebelum dan sesudah, sebagai bukti konversi masuk akal).
**File:** `audit_timestamp.csv`, `audit_timestamp_invalid.csv`, `log_konversi_gas.csv` di `OUT_DIR`.

In [ ]:
# ==== 2.2 BACA 27 STASIUN + AUDIT ====
files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_AQMS.csv"), recursive=True))
assert len(files) == 27, f"file AQMS harus 27, ditemukan {len(files)}"
raw0 = pd.concat([baca_stasiun(f) for f in files], ignore_index=True)
assert pd.Series([a["station"] for a in AUDIT_BACA]).is_unique, "ada >1 file untuk stasiun yang sama"
assert not raw0.duplicated(["station", "time"]).any(), "duplikat station+time setelah agregasi"

rpt_ts = pd.DataFrame(AUDIT_BACA).set_index("station").sort_index()
rpt_ts.to_csv(os.path.join(OUT_DIR, "audit_timestamp.csv"), index=True)
pd.DataFrame(AUDIT_TIMESTAMP_INVALID).to_csv(os.path.join(OUT_DIR, "audit_timestamp_invalid.csv"), index=False)
print("Validasi timestamp dan agregasi sumber:")
display(rpt_ts)
_terdeteksi = sorted(rpt_ts.index[(rpt_ts["n_geser_jam_vs_floor_start"] >= 0.99 * rpt_ts["n_baris_mentah"]) &
                                  (rpt_ts["n_baris_mentah"] > 0)])
print("Stasiun dengan midpoint yang mengubah jam relatif ke floor(timestamp_start):", _terdeteksi)
if set(_terdeteksi) != set(STASIUN_KOREKSI):
    print("[CATATAN] Daftar STASIUN_KOREKSI dan hasil deteksi midpoint berbeda; periksa audit_timestamp.csv.")

if KONVERSI_AKTIF:
    log_konversi = pd.DataFrame(LOG_KONVERSI)
    display(log_konversi)
    assert set(PPM_STATIONS) <= set(log_konversi.get("station", [])), "stasiun ppm belum terkonversi"
    tercatat = log_konversi.groupby("station")["gas"].apply(set)
    for st_ in PPM_STATIONS:
        assert set(GAS_WAJIB) <= tercatat.get(st_, set()), f"{st_}: gas belum lengkap terkonversi"
    assert not log_konversi.duplicated(["station", "gas"]).any()
    log_konversi.to_csv(os.path.join(OUT_DIR, "log_konversi_gas.csv"), index=False)

Validasi timestamp dan agregasi sumber:


,tz_mentah,contoh_timestamp_start,n_baris_mentah,n_slot_jam_terbentuk,n_jam_dengan_multi_pengukuran,n_pengukuran_tambahan_di_jam,n_durasi_bukan_1jam,n_geser_jam_vs_floor_start,n_timestamp_interval_tidak_valid,n_timestamp_di_luar_periode,n_qc_tidak_valid,n_qc_valid,n_sentinel_co
station,,,,,,,,,,,,,
BGL-01,naive,2024-01-01 00:02:10.000,24432,24432,0,0,0,0,0,0,125,24307,0
BGR-01,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,744,23688,0
BGR-02,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,737,23695,0
BJB-01,naive,2024-01-01 00:01:15.420,24432,24432,0,0,0,0,0,0,0,24432,0
BTM-01,naive,2023-12-31 23:48:15.100,24432,24432,0,0,0,24432,0,0,154,24278,0
IDP-01,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,110,24322,0
JKT-01,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,978,23454,0
JKT-02,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,952,23480,0
JKT-03,naive,2024-01-01 00:00:00,24432,24432,0,0,0,0,0,0,962,23470,0


Stasiun dengan midpoint yang mengubah jam relatif ke floor(timestamp_start): ['BTM-01', 'JMB-01', 'MLG-01', 'PKN-01', 'STG-01']


,station,gas,unit_sebelum,unit_sesudah,MW,vol_molar_L,median_sebelum,median_sesudah
0,BJB-01,so2,ppm,ug/m3,64.06,24.45,0.0032,8.384131
1,BJB-01,co,ppm,ug/m3,28.01,24.45,0.4750,544.161554
2,BJB-01,o3,ppm,ug/m3,48.00,24.45,0.0090,17.668712
3,BJB-01,no2,ppm,ug/m3,46.01,24.45,0.0164,30.861513
4,MDN-01,so2,ppm,ug/m3,64.06,24.45,0.0034,8.908139
5,MDN-01,co,ppm,ug/m3,28.01,24.45,0.5640,646.120245
6,MDN-01,o3,ppm,ug/m3,48.00,24.45,0.0130,25.521472
7,MDN-01,no2,ppm,ug/m3,46.01,24.45,0.0185,34.813292
8,PSW-01,so2,ppm,ug/m3,64.06,24.45,0.0029,7.598119
9,PSW-01,co,ppm,ug/m3,28.01,24.45,0.3690,422.727607


## 2.3 Grid jam lengkap dan deteksi sensor macet (flatline)
**Tujuan:** memastikan tiap stasiun punya baris untuk setiap jam, dan membuang pembacaan sensor yang macet.
**Proses:**
- membentuk grid 27 stasiun × 24.432 jam (slot tanpa data ditandai `slot_kosong`);
- PM2.5/PM10 stasiun `PM_EXCLUDE` (JMB-02 dan tiga stasiun ppm) dikosongkan;
- nilai identik ≥ 6 jam berturut-turut ditandai macet dan dikosongkan.
**Hasil:** tabel `raw` (659.664 baris), `audit_flatline_runs.csv`. Temuan: BJB-01 punya 1 run di mana blok polutan membeku serempak sementara cuaca tetap bergerak (±30% jam stasiun itu ditandai macet); stasiun lain tidak ada yang melewati ambang peringatan.

In [ ]:
# ==== 2.3 GRID JAM LENGKAP + FLATLINE ====
raw0 = raw0[(raw0["time"] >= pd.Timestamp(PERIODE[0])) &
            (raw0["time"] < pd.Timestamp(PERIODE[1]) + pd.Timedelta(days=1))].copy()
raw0["ada_baris"] = True
stasiun_all = sorted(META)
assert set(raw0["station"]) <= set(stasiun_all)
idx_jam = pd.date_range(pd.Timestamp(PERIODE[0]), pd.Timestamp(PERIODE[1]) + pd.Timedelta(hours=23), freq="h")
full_idx = pd.MultiIndex.from_product([stasiun_all, idx_jam], names=["station", "time"])
raw = raw0.set_index(["station", "time"]).reindex(full_idx)
raw["slot_kosong"] = raw["ada_baris"].isna()
raw["qc_ok"] = raw["qc_ok"].fillna(False).astype(bool)
raw["is_sentinel_co"] = raw["is_sentinel_co"].fillna(False).astype(bool)
count_cols = [c for c in raw.columns if c.startswith("n_") and c.endswith("_sumber")]
for c_ in count_cols:
    raw[c_] = raw[c_].fillna(0).astype(int)
raw = raw.drop(columns=["ada_baris"]).reset_index()

# Flatline: simpan flag sebelum nilai polutan di-NaN-kan.
PM_EXCLUDE = sorted(set(PM_EXCLUDE) | set(PPM_STATIONS))
for c in ("pm25", "pm10"):
    raw.loc[raw["station"].isin(PM_EXCLUDE), c] = np.nan

def flag_flatline(s, n):
    grp = s.ne(s.shift()).cumsum()
    return s.notna() & (s.groupby(grp).transform("size") >= n)

for p in POL:
    raw[f"stuck_{p}"] = raw.groupby("station")[p].transform(lambda s: flag_flatline(s, FLATLINE_JAM)).astype(bool)
raw["is_stuck"] = raw[[f"stuck_{p}" for p in POL]].any(axis=1)
raw["n_var_stuck"] = raw[[f"stuck_{p}" for p in POL]].sum(axis=1)

def ringkas_run_flatline(df):
    out = []
    for st, gg in df.groupby("station"):
        gg = gg.reset_index(drop=True)
        tersedia = [p for p in POL if gg[p].notna().any()]
        m = gg["is_stuck"]
        for _, rr in gg[m].groupby(m.ne(m.shift()).cumsum()[m]):
            aktif = [p for p in POL if rr[f"stuck_{p}"].any()]
            serempak = bool(tersedia) and set(tersedia) <= set(aktif)
            cuaca_gerak = bool(rr["temp"].nunique() > 1 or rr["rh"].nunique() > 1)
            jenis = ("blok polutan beku serempak, cuaca tetap bergerak -> indikasi cacat sumber/logger"
                     if serempak and cuaca_gerak and len(rr) > FLATLINE_JAM
                     else "satu variabel -> cek resolusi/batas deteksi sensor" if len(aktif) == 1
                     else "sebagian variabel -> tinjau manual")
            out.append(dict(station=st, mulai=rr["time"].min(), selesai=rr["time"].max(), n_jam=len(rr),
                            variabel_beku=",".join(aktif), serempak_semua_variabel=serempak,
                            nilai_beku="; ".join(f"{p}={rr[p].iloc[0]:.4g}" for p in aktif),
                            n_unik_suhu=int(rr["temp"].nunique()), n_qc_valid=int(rr["qc_ok"].sum()), jenis=jenis))
    return pd.DataFrame(out, columns=["station", "mulai", "selesai", "n_jam", "variabel_beku", "serempak_semua_variabel",
                                      "nilai_beku", "n_unik_suhu", "n_qc_valid", "jenis"])

audit_flatline_runs = ringkas_run_flatline(raw)
audit_flatline_runs.to_csv(os.path.join(OUT_DIR, "audit_flatline_runs.csv"), index=False)
print(f"Run flatline (>= {FLATLINE_JAM} jam identik): {len(audit_flatline_runs)} run")
display(audit_flatline_runs)
for st_, v_ in raw.groupby("station")["is_stuck"].mean().loc[lambda s: s > 0.10].items():
    print(f"[!] {st_}: {v_:.1%} jam ditandai flatline; tinjau audit_flatline_runs.csv.")
for p in POL:
    raw.loc[raw[f"stuck_{p}"], p] = np.nan

/tmp/ipykernel_476/14922737.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw["qc_ok"] = raw["qc_ok"].fillna(False).astype(bool)
/tmp/ipykernel_476/14922737.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw["is_sentinel_co"] = raw["is_sentinel_co"].fillna(False).astype(bool)


Run flatline (>= 6 jam identik): 1 run


,station,mulai,selesai,n_jam,variabel_beku,serempak_semua_variabel,nilai_beku,n_unik_suhu,n_qc_valid,jenis
0,BJB-01,2024-09-07,2025-07-09 08:00:00,7329,"so2,co,o3,no2",True,so2=39.3; co=2807; o3=15.71; no2=65.86,103,7329,"blok polutan beku serempak, cuaca tetap berger..."


[!] BJB-01: 30.0% jam ditandai flatline; tinjau audit_flatline_runs.csv.


## 2.4 Audit pembersihan per stasiun
**Tujuan:** merekap berapa baris yang masuk, lolos QC, tidak valid, sentinel, dan macet per stasiun.
**Hasil:** tabel `audit_cleaning` (dipakai sebagai sheet `audit_cleaning`) dan pemeriksaan bahwa grid tepat 659.664 baris tanpa duplikat.

In [ ]:
# ==== 2.4 AUDIT PEMBERSIHAN ====
# Audit sumber harus membedakan jumlah baris asli, QC, dan jumlah slot jam.
raw["qc_tidak_valid"] = raw["n_qc_tidak_valid_sumber"].fillna(0).gt(0) & (~raw["slot_kosong"])
audit_cleaning = raw.groupby("station").agg(
    n_slot=("time", "size"), n_slot_kosong=("slot_kosong", "sum"),
    n_pengukuran_sumber=("n_baris_sumber", "sum"),
    n_qc_valid_sumber=("n_qc_valid_sumber", "sum"),
    n_qc_tidak_valid=("n_qc_tidak_valid_sumber", "sum"),
    n_jam_dengan_minimal_satu_qc_buruk=("qc_tidak_valid", "sum"),
    n_sentinel_co=("n_sentinel_co_sumber", "sum"), n_jam_stuck=("is_stuck", "sum")).reset_index()
audit_cleaning.to_csv(os.path.join(OUT_DIR, "audit_cleaning_per_stasiun.csv"), index=False)
assert len(raw) == len(stasiun_all) * len(idx_jam) == EXPECTED_ROWS
assert raw["station"].nunique() == 27
assert raw["time"].min() == pd.Timestamp(PERIODE[0]) and raw["time"].max() == pd.Timestamp(PERIODE[1]) + pd.Timedelta(hours=23)
assert not raw.duplicated(["station", "time"]).any()
print(f"Full grid: {len(raw):,} baris; PLB-01 sumber {int(rpt_ts.loc['PLB-01','n_baris_mentah']):,} pengukuran -> {int(rpt_ts.loc['PLB-01','n_slot_jam_terbentuk']):,} slot jam.")

Full grid: 659,664 baris; PLB-01 sumber 36,381 pengukuran -> 24,432 slot jam.


## 2.5 Pemeriksaan zona waktu dan penyelesaian tabel jam-an
**Tujuan:** memastikan jam pada data konsisten dengan waktu lokal.
**Proses:** fase puncak suhu harian tiap stasiun dihitung dan dibandingkan dengan mediannya (median seluruh stasiun: pukul 13,00). Ini hanya pemeriksaan kewajaran, bukan bukti zona waktu; zona WIB mengacu pada booklet case study.
**Hasil:** `audit_zona_waktu.csv`, kolom bantu `date`, `hour`, `dow`, `kawasan`, dan tabel `raw` final.

In [ ]:
# ==== 2.5 CEK ZONA WAKTU + FINALISASI ====
# Pemeriksaan kewajaran fase suhu tidak membuktikan zona waktu.
_H = np.arange(24)
def _fase_jam(v):
    a = v - v.mean()
    return (np.arctan2((a * np.sin(2 * np.pi * _H / 24)).sum(), (a * np.cos(2 * np.pi * _H / 24)).sum()) * 24 / (2 * np.pi)) % 24
_tt = raw.dropna(subset=["temp"]).assign(_jam=lambda d: d["time"].dt.hour)
_prof = _tt.groupby(["station", "_jam"])["temp"].mean().unstack().reindex(columns=range(24))
audit_zona = pd.DataFrame({"n_jam_temp_valid": _tt.groupby("station").size()}).reindex(sorted(META))
audit_zona["fase_puncak_suhu_jam"] = [round(_fase_jam(_prof.loc[s].interpolate(limit_direction="both").to_numpy()), 2)
    if s in _prof.index and _prof.loc[s].notna().sum() >= 20 else np.nan for s in audit_zona.index]
_med = audit_zona.loc[audit_zona["n_jam_temp_valid"] >= 2000, "fase_puncak_suhu_jam"].median()
VERDIK_ZONA = ("konsisten dengan waktu lokal (WIB/WITA/WIT), bukan bukti zona tertentu" if 10 <= _med <= 16
               else "perlu peninjauan; pola suhu tidak cukup untuk memastikan zona waktu")
audit_zona["menyimpang_>3jam_dari_median"] = (audit_zona["fase_puncak_suhu_jam"] - _med).abs() > 3
audit_zona.reset_index().to_csv(os.path.join(OUT_DIR, "audit_zona_waktu.csv"), index=False)
print(f"Fase puncak suhu median: {_med:.2f} jam -> {VERDIK_ZONA}")
if not ZONA_DIDUKUNG_DOKUMENTASI:
    print(f"[!] {KONFIRMASI_ZONA_NAIVE} tetap asumsi: metadata tidak memberi konfirmasi timezone independen.")

raw["periode_anomali"] = raw["station"].isin(ANOMALI_STASIUN) & (raw["time"] >= pd.Timestamp(ANOMALI_MULAI))
pass  # CSV jam-an (~120 MB) dilewati; tidak dibutuhkan Looker
audit_cleaning = audit_cleaning.merge(
    raw.groupby("station")["pm25"].count().rename("n_jam_pm25_valid_akhir").reset_index(), on="station")
audit_cleaning.to_csv(os.path.join(OUT_DIR, "audit_cleaning_per_stasiun.csv"), index=False)

# Ringkasan khusus slot dengan lebih dari satu pengukuran sumber.
plb = raw[raw["station"].eq("PLB-01")]
cols_plb = ["station", "time", "n_baris_sumber", "n_qc_valid_sumber", "n_qc_tidak_valid_sumber",
            "pm25", "pm10", "so2", "co", "o3", "no2", "temp", "rh", "ws", "wd"]
plb.loc[plb["n_baris_sumber"].gt(1), cols_plb].to_csv(os.path.join(OUT_DIR, "audit_pengukuran_multi_PLB01.csv"), index=False)

_gas = [c for c in POL if c not in ("pm25", "pm10")]
print("Median gas (PPM_STATIONS vs lainnya):")
display(raw.assign(grup=np.where(raw["station"].isin(PPM_STATIONS), "asal_ppm_sudah_dikonversi", "lain"))
        .groupby("grup")[_gas].median().round(2))
raw["date"] = raw["time"].dt.normalize()
raw["hour"] = raw["time"].dt.hour
raw["dow"] = raw["time"].dt.dayofweek
raw["kawasan"] = raw["station"].map(lambda s: META[s]["kawasan"])
TOTAL_HARI = (pd.Timestamp(PERIODE[1]) - pd.Timestamp(PERIODE[0])).days + 1
ANOM_T = pd.Timestamp(ANOMALI_MULAI)

Fase puncak suhu median: 13.00 jam -> konsisten dengan waktu lokal (WIB/WITA/WIT), bukan bukti zona tertentu
Median gas (PPM_STATIONS vs lainnya):


,so2,co,o3,no2
grup,,,,
asal_ppm_sudah_dikonversi,7.86,508.65,17.67,27.1
lain,8.40,379.70,14.00,19.6


# 3. Agregasi harian
**Tujuan:** mengubah data jam-an menjadi rata-rata harian per stasiun.
**Aturan:** sebuah hari valid bila memiliki ≥ 18 jam valid (`MIN_JAM_VALID`); arah angin dirata-ratakan sirkular.
**Seleksi stasiun PM2.5:** stasiun masuk `PM_COMPARE` bila bukan `PM_EXCLUDE` dan ≥ 60% hari punya PM2.5 valid. Hasil: **22 stasiun**. Lima stasiun keluar dengan coverage PM2.5 ≈ 0%: BJB-01, MDN-01, PSW-01 (stasiun ppm, tanpa PM), JMB-02, dan MER-01 (hanya 2 hari).
**Hasil:** tabel `daily`, daftar `PM_COMPARE`.

In [ ]:
# ============================================================
# ==== 3. AGREGASI HARIAN + SELEKSI STASIUN PM2.5 ====
# ============================================================
_v = [c for c in POL + CUACA if c != "wd"]
_g = raw.groupby(["station", "date"])
daily = _g[_v].mean().where(_g[_v].count() >= MIN_JAM_VALID)

# arah angin: rata-rata sirkular (bukan rata-rata biasa)
_r = np.deg2rad(raw["wd"])
_t = raw.assign(_s=np.sin(_r), _c=np.cos(_r)).groupby(["station", "date"])
_wd = np.rad2deg(np.arctan2(_t["_s"].mean(), _t["_c"].mean())) % 360
daily["wd"] = _wd.where(raw.groupby(["station", "date"])["wd"].count() >= MIN_JAM_VALID)
daily = daily.reset_index()
daily["kawasan"] = daily["station"].map(lambda s: META[s]["kawasan"])
daily["month"] = daily["date"].dt.month
daily["ym"] = daily["date"].dt.to_period("M").dt.to_timestamp()

# stasiun PM yang layak: bukan PM_EXCLUDE dan >= 60% hari punya PM2.5 valid
MIN_COV_PM = 0.60
cov_pm = daily.groupby("station")["pm25"].count() / TOTAL_HARI
PM_COMPARE = sorted(s for s in META if s not in PM_EXCLUDE and cov_pm.get(s, 0) >= MIN_COV_PM)
PM_MAIN = list(PM_COMPARE)
print("Coverage hari PM2.5 per stasiun:"); print(cov_pm.round(3).sort_values().to_string())
print("\nPM_COMPARE:", len(PM_COMPARE), "stasiun | MER-01 di PM_COMPARE?", "MER-01" in PM_COMPARE)

Coverage hari PM2.5 per stasiun:
station
BJB-01    0.000
JMB-02    0.000
MDN-01    0.000
PSW-01    0.000
MER-01    0.002
PLB-01    0.684
SM2-01    0.757
BTM-01    1.000
JKT-02    1.000
JKT-01    1.000
IDP-01    1.000
JKT-03    1.000
KTB-01    1.000
BGL-01    1.000
BGR-01    1.000
BGR-02    1.000
KTG-01    1.000
KTG-02    1.000
JMB-01    1.000
MLG-01    1.000
PKU-01    1.000
PKN-01    1.000
MRS-01    1.000
PLU-01    1.000
SRG-01    1.000
STG-01    1.000
YKY-01    1.000

PM_COMPARE: 22 stasiun | MER-01 di PM_COMPARE? False


# 4. Deteksi anomali dan aturan pengecualian
## 4.1 Persiapan patch V8
**Tujuan:** menetapkan jendela dugaan anomali (8–14 Okt 2026) dan memastikan variabel hasil bagian sebelumnya tersedia.

In [ ]:
# ==== 4.1 JENDELA DUGAAN ANOMALI ====
DUGAAN_MULAI = pd.Timestamp("2026-10-08")
DUGAAN_AKHIR = pd.Timestamp("2026-10-14")


# cek variabel yang dibutuhkan
for _v in ['raw','daily','META','PM_COMPARE','HARI','OUT_DIR','ANOMALI_STASIUN','ANOM_T']:
    assert _v in globals(), f'Variabel {_v} belum ada: jalankan Cell 0-3 notebook v7 dulu.'
print('Variabel lengkap.')

Variabel lengkap.


## 4.2 Audit otomatis dugaan anomali 8–14 Okt 2026
**Tujuan:** mendeteksi stasiun yang pola 7 hari terakhirnya mencurigakan, tanpa menilai manual.
**Proses:** untuk tiap stasiun PM2.5, bandingkan hari pertama jendela dengan median 14 hari sebelumnya, lalu periksa pola:
- **dugaan ramp** — lompat > 20% lalu naik monoton dengan kenaikan harian hampir konstan (CV < 0,25);
- **dugaan datar** — lompat > 20% lalu nyaris konstan (SD relatif < 1%);
- **data kurang** — hari valid sebelum jendela < 5 atau di jendela < 5, atau jeda antar observasi > 2 hari (tidak dikecualikan).
**Hasil:** 7 stasiun dugaan (BTM-01, JMB-01, KTB-01, PKN-01, PKU-01, STG-01 = ramp; PLB-01 = datar); SM2-01 "data kurang". Penanda ini untuk ditinjau, bukan bukti galat.

In [ ]:
# ==== 4.2 AUDIT DUGAAN ANOMALI ====
MIN_HARI_WINDOW = 5      # minimal 5 dari 7 hari bertanggal 8-14 Okt harus punya PM2.5 valid
MAKS_JEDA_HARI = 2       # selisih tanggal berurutan maksimum (hari) antar observasi valid

def audit_dugaan_anomali(daily, mulai=DUGAAN_MULAI, akhir=DUGAAN_AKHIR, pra_hari=14, ambang=0.20):
    rows = []
    for s in PM_COMPARE:
        d = daily[daily["station"] == s].set_index("date")["pm25"].sort_index()   # urut tanggal dulu
        pra = d[(d.index >= mulai - pd.Timedelta(days=pra_hari)) & (d.index < mulai)].dropna()
        win = d[(d.index >= mulai) & (d.index <= akhir)].dropna()
        n_win = int(len(win))
        jeda = int(win.index.to_series().diff().dt.days.max()) if n_win >= 2 else np.nan
        dasar = dict(station=s, n_pra=len(pra), n_hari_valid_window=n_win, maks_jeda_hari=jeda)
        if len(pra) < 5 or n_win < MIN_HARI_WINDOW:
            rows.append({**dasar, "dugaan": "data kurang",
                         "alasan": f"hari valid pra={len(pra)} (min 5), window={n_win} (min {MIN_HARI_WINDOW})"})
            continue
        if jeda > MAKS_JEDA_HARI:
            rows.append({**dasar, "dugaan": "data kurang", "alasan": f"jeda antar observasi {jeda} hari (maks {MAKS_JEDA_HARI})"})
            continue
        base = pra.median()
        loncat = win.iloc[0] / base - 1                               # perubahan hari pertama vs median sebelumnya
        diff = win.diff().dropna()
        monoton_naik = bool((diff > 0).all())
        rel_sd = float(win.std() / win.mean())                        # ~0 berarti datar
        diff_cv = float(diff.std() / diff.mean()) if monoton_naik and diff.mean() != 0 else np.nan
        linear = monoton_naik and diff_cv < 0.25                      # kenaikan harian nyaris konstan
        datar = rel_sd < 0.01
        if abs(loncat) > ambang and linear:
            jenis, alasan = "dugaan ramp", f"lompat {loncat:+.0%} lalu naik monoton, CV kenaikan harian {diff_cv:.2f}"
        elif abs(loncat) > ambang and datar:
            jenis, alasan = "dugaan datar", f"lompat {loncat:+.0%} lalu hampir konstan, SD relatif {rel_sd:.3f}"
        else:
            jenis, alasan = "normal", "tidak memenuhi pola ramp/datar"
        rows.append({**dasar, "median_pra": base, "perubahan_hari_pertama": loncat,
                     "kenaikan_harian_rata2": diff.mean(), "cv_kenaikan_harian": diff_cv,
                     "rel_sd_window": rel_sd, "dugaan": jenis, "alasan": alasan})
    return pd.DataFrame(rows)

audit_dugaan = audit_dugaan_anomali(daily)
display(audit_dugaan.round(3))
audit_dugaan.to_csv(os.path.join(OUT_DIR, "audit_dugaan_anomali_okt2026.csv"), index=False)
# CATATAN: ini penanda untuk ditinjau (bukan bukti galat). Cek juga data jam-an stasiun yang kena
# dan booklet case study sebelum memutuskan dibuang. KTG sudah ditandai oleh aturan lama.
# Hanya "dugaan ramp" dan "dugaan datar" yang dikecualikan; "data kurang" TIDAK dikecualikan.
DUGAAN_STASIUN = audit_dugaan.loc[audit_dugaan["dugaan"].isin(["dugaan ramp", "dugaan datar"]), "station"].tolist()
print("Dugaan anomali 8-14 Okt 2026:", DUGAAN_STASIUN)
print("Data kurang (tidak dikecualikan, perlu cek manual):", audit_dugaan.loc[audit_dugaan["dugaan"] == "data kurang", "station"].tolist())
assert set(DUGAAN_STASIUN) <= set(PM_COMPARE)

,station,n_pra,n_hari_valid_window,maks_jeda_hari,median_pra,perubahan_hari_pertama,kenaikan_harian_rata2,cv_kenaikan_harian,rel_sd_window,dugaan,alasan
0,BGL-01,14,7,1,32.069,-0.034,0.313,NaN,0.026,normal,tidak memenuhi pola ramp/datar
1,BGR-01,14,7,1,12.998,0.010,0.100,NaN,0.034,normal,tidak memenuhi pola ramp/datar
2,BGR-02,14,7,1,19.258,-0.036,0.160,NaN,0.018,normal,tidak memenuhi pola ramp/datar
3,BTM-01,14,7,1,100.577,-0.347,16.510,0.072,0.314,dugaan ramp,"lompat -35% lalu naik monoton, CV kenaikan har..."
4,IDP-01,14,7,1,8.721,-0.012,0.068,NaN,0.042,normal,tidak memenuhi pola ramp/datar
5,JKT-01,14,7,1,52.137,-0.020,-0.038,NaN,0.021,normal,tidak memenuhi pola ramp/datar
6,JKT-02,14,7,1,64.130,0.011,-0.026,NaN,0.007,normal,tidak memenuhi pola ramp/datar
7,JKT-03,14,7,1,58.219,-0.000,-0.129,NaN,0.012,normal,tidak memenuhi pola ramp/datar
8,JMB-01,14,7,1,111.677,-0.444,20.233,0.056,0.361,dugaan ramp,"lompat -44% lalu naik monoton, CV kenaikan har..."
9,KTB-01,14,7,1,51.571,-0.241,24.977,0.021,0.472,dugaan ramp,"lompat -24% lalu naik monoton, CV kenaikan har..."


Dugaan anomali 8-14 Okt 2026: ['BTM-01', 'JMB-01', 'KTB-01', 'PKN-01', 'PKU-01', 'PLB-01', 'STG-01']
Data kurang (tidak dikecualikan, perlu cek manual): ['SM2-01']


## 4.3 Satu aturan pengecualian + status berlapis
**Tujuan:** menjamin semua keluaran memakai pengecualian yang sama.
**Fungsi `tanpa_semua_anomali(df)`:** membuang baris KTG-01/KTG-02 ≥ 1 Sep 2026 **dan** baris stasiun dugaan pada 8–14 Okt 2026.
**Kolom `status_data`** (berlapis, kelayakan dasar dicek lebih dulu): `normal`, `stasiun tidak dipakai untuk PM2.5`, `PM2.5 tidak valid`, `anomali KTG`, `dugaan anomali`.
**Kolom `pm25_dipakai`:** PM2.5 harian hanya bila status `normal`.
**Hasil:** 21.691 hari normal; **136 hari-stasiun PM2.5 dikecualikan** (88 KTG + 48 dugaan).

In [ ]:
# ==== 4.3 ATURAN PENGECUALIAN TUNGGAL + STATUS ====
def mask_anomali(df, kolom_tanggal):
    """True untuk baris yang dikecualikan: KTG >= 1 Sep 2026 atau dugaan ramp/datar 8-14 Okt 2026."""
    t = pd.to_datetime(df[kolom_tanggal])
    ktg = df["station"].isin(ANOMALI_STASIUN) & (t >= ANOM_T)
    dug = df["station"].isin(DUGAAN_STASIUN) & (t >= DUGAAN_MULAI) & (t <= DUGAAN_AKHIR)
    return ktg | dug

def tanpa_semua_anomali(df, kolom_tanggal="date"):
    return df[~mask_anomali(df, kolom_tanggal)].copy()

# status berlapis: kelayakan dasar dulu, baru pengecualian anomali (tidak menimpa status dasar)
t_ = pd.to_datetime(daily["date"])
ktg_ = daily["station"].isin(ANOMALI_STASIUN) & (t_ >= ANOM_T)
dug_ = daily["station"].isin(DUGAAN_STASIUN) & (t_ >= DUGAAN_MULAI) & (t_ <= DUGAAN_AKHIR)
daily["status_data"] = "normal"
daily.loc[~daily["station"].isin(PM_COMPARE), "status_data"] = "stasiun tidak dipakai untuk PM2.5"
daily.loc[(daily["status_data"] == "normal") & daily["pm25"].isna(), "status_data"] = "PM2.5 tidak valid (jam valid < MIN_JAM_VALID)"
daily.loc[(daily["status_data"] == "normal") & ktg_, "status_data"] = "anomali KTG >= 1 Sep 2026 (belum terverifikasi)"
daily.loc[(daily["status_data"] == "normal") & dug_, "status_data"] = "dugaan anomali 8-14 Okt 2026 (perlu verifikasi)"
daily["pm25_dipakai"] = daily["pm25"].where(daily["status_data"] == "normal")
print(daily["status_data"].value_counts())
print(f"Hari-stasiun PM2.5 dikecualikan: {int(daily['status_data'].str.startswith(('anomali','dugaan')).sum())}")
assert daily.loc[~daily["station"].isin(PM_COMPARE), "pm25_dipakai"].isna().all()

status_data
normal                                             21691
stasiun tidak dipakai untuk PM2.5                   5090
PM2.5 tidak valid (jam valid < MIN_JAM_VALID)        569
anomali KTG >= 1 Sep 2026 (belum terverifikasi)       88
dugaan anomali 8-14 Okt 2026 (perlu verifikasi)       48
Name: count, dtype: int64
Hari-stasiun PM2.5 dikecualikan: 136


# 5. Pola jam × hari (untuk heatmap Looker)
**Tujuan:** melihat jam dan hari kapan PM2.5 tinggi.
**Proses:** data jam-an PM2.5 setelah `tanpa_semua_anomali`, dirata-ratakan per stasiun × hari-dalam-pekan × jam; lalu digabung bobot-stasiun-setara, dan per kawasan untuk hari kerja vs akhir pekan.
**Hasil (3 sheet):** `pola_jam_hari_stasiun` (bisa difilter per stasiun), `heatmap_jam_hari_v2` (gabungan), `pola_jam_kawasan`. Rentang rata-rata antarjam ≈ 33–48 µg/m³, sedangkan selisih antarhari hanya ≈ 0,1 µg/m³.

In [ ]:
# ==== 5. POLA JAM x HARI ====
raw_ok = tanpa_semua_anomali(raw[raw["station"].isin(PM_COMPARE)], "date").dropna(subset=["pm25"])
pola = (raw_ok.groupby(["station", "dow", "hour"])
              .agg(pm25_rata2=("pm25", "mean"), n_obs_valid=("pm25", "size"), n_hari=("date", "nunique")).reset_index())
pola["hari"] = pola["dow"].map(lambda i: HARI[i])
pola["hari_urut"] = pola["dow"] + 1
pola["jam_label"] = pola["hour"].map(lambda h: f"{h:02d}:00")
pola["kategori_kawasan"] = pola["station"].map(lambda s: META[s]["kawasan"])
pola.to_csv(os.path.join(OUT_DIR, "pola_jam_hari_stasiun.csv"), index=False)

# gabungan (bobot stasiun setara) - menggantikan heatmap_jam_hari lama
heatmap_v2 = (pola.groupby(["dow", "hari", "hari_urut", "hour", "jam_label"])
                  .agg(pm25_rata2=("pm25_rata2", "mean"), n_stasiun=("station", "nunique"),
                       n_obs_valid=("n_obs_valid", "sum"), n_hari_stasiun=("n_hari", "sum")).reset_index())
# n_obs_valid = jumlah jam-pengukuran valid; n_hari_stasiun = jumlah hari-stasiun yang berkontribusi
heatmap_v2.to_csv(os.path.join(OUT_DIR, "heatmap_jam_hari_v2.csv"), index=False)

# pola jam per kawasan, weekday vs weekend (hari kerja vs akhir pekan)
raw_ok = raw_ok.assign(jenis_hari=np.where(raw_ok["dow"] >= 5, "Akhir pekan", "Hari kerja"))
pjk = (raw_ok.groupby(["station", "jenis_hari", "hour"])["pm25"].mean().reset_index()
             .assign(kategori_kawasan=lambda d: d["station"].map(lambda s: META[s]["kawasan"]))
             .groupby(["kategori_kawasan", "jenis_hari", "hour"])["pm25"].mean().reset_index()
             .rename(columns={"pm25": "pm25_rata2"}))
pjk["jam_label"] = pjk["hour"].map(lambda h: f"{h:02d}:00")
pjk.to_csv(os.path.join(OUT_DIR, "pola_jam_kawasan.csv"), index=False)

print("Rentang rata-rata per jam (µg/m³):", heatmap_v2.groupby("hour")["pm25_rata2"].mean().agg(["min", "max"]).round(1).to_dict())
print("Selisih rata-rata per hari (maks-min):",
      round(float(heatmap_v2.groupby("hari")["pm25_rata2"].mean().agg(lambda s: s.max() - s.min())), 2))

Rentang rata-rata per jam (µg/m³): {'min': 33.4, 'max': 48.1}
Selisih rata-rata per hari (maks-min): 0.11


# 6. Fungsi analisis
Bagian ini hanya **mendefinisikan fungsi** (tidak menghitung apa pun). Tiga sel berikut menjelaskan fungsinya.
## 6.1 Fungsi korelasi (Spearman + block bootstrap)
**`hitung_korelasi(versi, d, B)`** menghitung korelasi Spearman antara 6 polutan dan 3 variabel cuaca (suhu, kelembapan, kecepatan angin) = 18 pasangan, pada 4 skop:
| Skop | Arti |
|---|---|
| `harian_gabungan` | semua stasiun-hari digabung (pembanding; bercampur musim) |
| `harian_dalam_stasiun` | nilai dikurangi rata-rata stasiunnya (membuang beda antarstasiun) |
| `harian_bebas_musim` | dikurangi rata-rata stasiun × bulan-kalender (membuang pola musiman; analisis utama) |
| `bulanan_dalam_stasiun` | rata-rata bulanan per stasiun (≥ 16 hari), dikurangi rata-rata stasiun |

**Ketidakpastian:** *block bootstrap* — 34 bulan-kalender diambil ulang dengan pengembalian sebanyak B kali (seed 42), sehingga ketergantungan antarhari dalam bulan yang sama terjaga. Dari sini diperoleh CI 95% per pasangan dan CI **simultan** (kontrol terhadap 18 uji sekaligus).

In [ ]:
# ==== 6.1 FUNGSI KORELASI ====
import numpy as np, pandas as pd, openpyxl
from scipy.stats import rankdata
from concurrent.futures import ProcessPoolExecutor
SKOP = ["harian_gabungan", "harian_dalam_stasiun", "harian_bebas_musim", "bulanan_dalam_stasiun"]
POLK = ["pm25", "pm10", "so2", "co", "o3", "no2"]
VARK = ["temp", "rh", "ws"]
MIN_HARI_BULAN = 16

def _demean(x, g):
    return x - x.groupby(g).transform("mean")

def _spearman(X):
    r = rankdata(X, axis=0); r = r - r.mean(0)
    return (r[:, 0] * r[:, 1]).sum() / np.sqrt((r[:, 0] ** 2).sum() * (r[:, 1] ** 2).sum())

def hitung_korelasi(versi, d, B, seed=42):
    """d = tabel harian (sudah di-mask sesuai versi). Block bootstrap per bulan-kalender."""
    d = d.copy()
    d["ym"] = d["date"].dt.to_period("M"); d["moy"] = d["date"].dt.month
    blocks = sorted(d["ym"].unique()); NB = len(blocks)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, NB, size=(B, NB))
    rows = []
    for sc in SKOP:
        S = {}
        for p in POLK:
            for v in VARK:
                if sc == "bulanan_dalam_stasiun":
                    x = d[["station", "ym", p, v]].dropna()
                    g = x.groupby(["station", "ym"]).agg(a=(p, "mean"), b=(v, "mean"), nd=(p, "size")).reset_index()
                    g = g[g["nd"] >= MIN_HARI_BULAN]
                    g["a"] = _demean(g["a"], g["station"]); g["b"] = _demean(g["b"], g["station"])
                    S[(p, v)] = (g["ym"].to_numpy(), g[["a", "b"]].to_numpy())
                    continue
                x = d[["station", "ym", "moy", p, v]].dropna()
                if sc == "harian_gabungan":
                    A = x[[p, v]].to_numpy()
                elif sc == "harian_dalam_stasiun":
                    A = np.c_[_demean(x[p], x["station"]), _demean(x[v], x["station"])]
                else:
                    gg = [x["station"], x["moy"]]
                    A = np.c_[_demean(x[p], gg), _demean(x[v], gg)]
                S[(p, v)] = (x["ym"].to_numpy(), A)
        keys = list(S)
        est = np.array([_spearman(S[k][1]) for k in keys])
        idx = {}
        for k in keys:
            ymv = S[k][0]; order = np.argsort(ymv, kind="stable"); ys = ymv[order]
            idx[k] = (order, {b: (np.searchsorted(ys, b, "left"), np.searchsorted(ys, b, "right")) for b in blocks})
        boot = np.empty((B, len(keys)))
        for bi in range(B):
            for j, k in enumerate(keys):
                order, st = idx[k]
                sel = np.concatenate([order[st[blocks[t]][0]:st[blocks[t]][1]] for t in draws[bi]])
                boot[bi, j] = _spearman(S[k][1][sel])
        lo = np.nanpercentile(boot, 2.5, axis=0); hi = np.nanpercentile(boot, 97.5, axis=0)
        c = np.nanpercentile(np.nanmax(np.abs(boot - est), axis=1), 95)
        for j, k in enumerate(keys):
            rows.append(dict(versi=versi, scope=sc, polutan=k[0], variabel=k[1], rho=est[j], ci95_lo=lo[j], ci95_hi=hi[j],
                             n=len(S[k][1]), n_blok=NB, c_simultan=c, ci_sim_lo=est[j] - c, ci_sim_hi=est[j] + c,
                             m_pasangan_keluarga=len(keys)))
        print(f"  [{versi}] {sc} selesai (c_simultan={c:.4f})", flush=True)
    return pd.DataFrame(rows)

## 6.2 Fungsi tabel weekday/weekend dan episode
- **`tabel_weekday_weekend`** — rata-rata polutan hari kerja vs akhir pekan per kawasan (bobot-stasiun-setara: rata-rata per stasiun dulu, baru dirata-ratakan antarstasiun), memakai `tanpa_semua_anomali` untuk **semua polutan**.
- **`tabel_episode`** — episode PM2.5 = ≥ 3 hari kalender berturut-turut dengan rata-rata harian > 55,4 µg/m³. Bagian yang dikecualikan tetap ditampilkan sebagai baris terpisah dengan `ada_nilai_belum_tervalidasi = True` (filter False untuk angka resmi).

In [ ]:
# ==== 6.2 FUNGSI WEEKDAY/WEEKEND + EPISODE ====
def kategori_ispu(x):
    return ("baik" if x <= 15.5 else "sedang" if x <= 55.4 else "tidak sehat" if x <= 150.4
            else "sangat tidak sehat" if x <= 250.4 else "berbahaya")

def tabel_weekday_weekend(d, mask, PM_COMPARE):
    d = d[~mask].copy()
    d["wk"] = np.where(d["date"].dt.dayofweek >= 5, "Weekend", "Weekday")
    LAB = {"co": "CO", "no2": "NO2", "o3": "O3", "pm10": "PM10", "pm25": "PM2.5", "so2": "SO2"}
    rows = []
    for col in ["co", "no2", "o3", "pm10", "pm25", "so2"]:
        x = d[d["station"].isin(PM_COMPARE)] if col in ("pm25", "pm10") else d
        x = x.dropna(subset=[col])
        a = x.groupby(["kawasan", "station", "wk"])[col].mean().reset_index().groupby(["kawasan", "wk"])[col].mean().unstack()
        n = x.groupby(["kawasan", "wk"]).size().unstack(); ns = x.groupby(["kawasan", "wk"])["station"].nunique().unstack()
        for k in sorted(a.index):
            rows.append([LAB[col], k, int(n.loc[k, "Weekday"]), int(n.loc[k, "Weekend"]), int(ns.loc[k, "Weekday"]), int(ns.loc[k, "Weekend"]),
                         a.loc[k, "Weekday"], a.loc[k, "Weekend"], a.loc[k, "Weekend"] / a.loc[k, "Weekday"] - 1])
    return pd.DataFrame(rows, columns=["polutan", "kategori_kawasan", "n_hari_Weekday", "n_hari_Weekend", "n_stasiun_Weekday",
                                       "n_stasiun_Weekend", "nilai_Weekday", "nilai_Weekend", "selisih_pct_weekend_vs_weekday"])

def tabel_episode(d, mask_ktg, mask_dug, PM_COMPARE, prov, kaw, ambang=55.4, min_hari=3):
    def episodes(x, status, flag):
        out = []
        for st, g in x.groupby("station"):
            g = g.sort_values("date"); g = g[g["val"] > ambang]
            if g.empty: continue
            run = (g["date"].diff().dt.days != 1).cumsum()
            for _, r in g.groupby(run):
                if len(r) >= min_hari:
                    out.append(dict(station_id=st, provinsi=prov[st], mulai=r["date"].min(), selesai=r["date"].max(), durasi_hari=len(r),
                                    pm25_rata2=r["val"].mean(), pm25_maks_harian=r["val"].max(),
                                    kategori_ambang_ISPU_pada_maks=kategori_ispu(r["val"].max()), status_validasi=status,
                                    ada_nilai_belum_tervalidasi=flag, kategori_kawasan=kaw[st]))
        return out
    x = d[d["station"].isin(PM_COMPARE)].copy(); x["val"] = x["pm25"].round(3)
    mk, md = mask_ktg.loc[x.index], mask_dug.loc[x.index]
    E = (episodes(x[~(mk | md)], "di luar periode anomali yang ditandai", False)
         + episodes(x[mk], "periode anomali KTG, belum terverifikasi (>= 1 Sep 2026)", True)
         + episodes(x[md], "dugaan anomali 8-14 Okt 2026, belum terverifikasi", True))
    e = pd.DataFrame(E).sort_values(["station_id", "mulai"]).reset_index(drop=True)
    e["mulai"] = e["mulai"].dt.strftime("%Y-%m-%d"); e["selesai"] = e["selesai"].dt.strftime("%Y-%m-%d")
    return e

## 6.3 Fungsi label korelasi dan catatan
**`finalkan_korelasi`** menambahkan kolom interpretasi:
- `kekuatan` — |ρ| < 0,1 sangat lemah; < 0,3 lemah; < 0,5 sedang; selainnya kuat;
- `tanda_konsisten_antar_scope` — arah korelasi (positif/negatif) sama di keempat skop;
- `indikasi_eksploratif` — *indikasi* bila CI simultan tidak memuat 0 **dan** tanda konsisten; *indikasi lemah* bila hanya CI 95%; selainnya *tidak ada indikasi*.

In [ ]:
# ==== 6.3 LABEL KORELASI ====
def finalkan_korelasi(K):
    K = K.copy()
    K["o1"] = K["versi"].map({"semua_data": 0, "tanpa_semua_anomali": 1}); K["o2"] = K["scope"].map(SKOP.index)
    K["o3"] = K["polutan"].map(POLK.index); K["o4"] = K["variabel"].map(VARK.index)
    K = K.sort_values(["o1", "o2", "o3", "o4"]).drop(columns=["o1", "o2", "o3", "o4"]).reset_index(drop=True)
    K["keluarga_uji"] = "per_scope"
    K["ci95_tidak_memuat_0"] = (K.ci95_lo > 0) | (K.ci95_hi < 0)
    K["ci_sim_tidak_memuat_0"] = (K.ci_sim_lo > 0) | (K.ci_sim_hi < 0)
    K["kekuatan"] = K["rho"].abs().map(lambda a: "sangat lemah" if a < .1 else "lemah" if a < .3 else "sedang" if a < .5 else "kuat")
    K["tanda_konsisten_antar_scope"] = K.groupby(["versi", "polutan", "variabel"])["rho"].transform(
        lambda r: bool((np.sign(r) == np.sign(r.iloc[0])).all()))
    K["indikasi_eksploratif"] = np.where(K.ci_sim_tidak_memuat_0 & K.tanda_konsisten_antar_scope, "indikasi (CI simultan + tanda konsisten)",
                                np.where(K.ci95_tidak_memuat_0 & K.tanda_konsisten_antar_scope, "indikasi lemah (CI95 saja)", "tidak ada indikasi"))
    return K[["versi", "scope", "polutan", "variabel", "rho", "ci95_lo", "ci95_hi", "n", "n_blok", "c_simultan", "ci_sim_lo", "ci_sim_hi",
              "m_pasangan_keluarga", "keluarga_uji", "ci95_tidak_memuat_0", "ci_sim_tidak_memuat_0", "kekuatan",
              "tanda_konsisten_antar_scope", "indikasi_eksploratif"]]

# 7. Hitung weekday/weekend dan episode
**Tujuan:** menjalankan dua fungsi di atas pada data harian.
**Mask pengecualian:** `mask_ktg` (KTG ≥ 1 Sep 2026) dan `mask_dug` (dugaan 8–14 Okt). Bila dihitung per baris stasiun-hari, mask mengenai 137 baris: 136 baris PM2.5 + 1 hari PLB-01 yang tidak punya PM2.5 valid tetapi punya data gas (ikut dibuang agar semua polutan konsisten).
**Hasil:** tabel `ww` (30 baris) dan `ep` (episode, termasuk baris yang dikecualikan).
**Catatan baca:** nilai gas kawasan pesisir bergantung pada MER-01 yang hanya punya 2 hari gas valid namun berbobot ⅓; jangan dibaca sebagai efek akhir pekan.

In [ ]:
# ==== 7. WEEKDAY/WEEKEND + EPISODE ====
d = daily.copy()
for c in ["pm25", "pm10", "so2", "co", "o3", "no2", "temp", "rh", "ws"]:
    d[c] = d[c].round(3)
mask_ktg = d["station"].isin(ANOMALI_STASIUN) & (d["date"] >= ANOM_T)
mask_dug = d["station"].isin(DUGAAN_STASIUN) & (d["date"] >= DUGAAN_MULAI) & (d["date"] <= DUGAAN_AKHIR)
mask = mask_ktg | mask_dug
print(f"Baris stasiun-hari dikecualikan: {int(mask.sum())}")

ww = tabel_weekday_weekend(d, mask, PM_COMPARE)
prov = {k: v["provinsi"] for k, v in META.items()}; kaw = {k: v["kawasan"] for k, v in META.items()}
ep = tabel_episode(d, mask_ktg, mask_dug, PM_COMPARE, prov, kaw)

print("\nSelisih akhir pekan vs hari kerja (%), PM2.5:")
print(ww[ww.polutan == "PM2.5"].assign(pct=lambda x: (x.selisih_pct_weekend_vs_weekday * 100).round(2))
      [["kategori_kawasan", "n_hari_Weekday", "n_hari_Weekend", "pct"]].to_string(index=False))
print(f"\nEpisode PM2.5: {len(ep)} baris, {ep.station_id.nunique()} stasiun")
display(ep[ep.mulai >= "2026-01-01"][["station_id", "mulai", "selesai", "durasi_hari", "pm25_maks_harian", "ada_nilai_belum_tervalidasi"]])

Baris stasiun-hari dikecualikan: 137

Selisih akhir pekan vs hari kerja (%), PM2.5:
     kategori_kawasan  n_hari_Weekday  n_hari_Weekend   pct
    background bersih            2907            1158  0.31
      gambut/karhutla            4782            1900  0.28
   industri/pelabuhan            2730            1088 -1.20
perkotaan/lalu lintas            3640            1450  0.73
              pesisir            1456             580 -0.05

Episode PM2.5: 76 baris, 12 stasiun


,station_id,mulai,selesai,durasi_hari,pm25_maks_harian,ada_nilai_belum_tervalidasi
2,BTM-01,2026-08-01,2026-10-07,68,103.370,False
3,BTM-01,2026-10-08,2026-10-14,7,164.708,True
7,JKT-03,2026-07-01,2026-10-14,106,60.084,False
11,JMB-01,2026-08-01,2026-10-07,68,113.692,False
12,JMB-01,2026-10-08,2026-10-14,7,183.483,True
13,KTB-01,2026-10-09,2026-10-14,6,188.992,True
18,KTG-01,2026-07-01,2026-08-31,62,96.708,False
19,KTG-01,2026-09-01,2026-10-14,44,1127.574,True
27,KTG-02,2026-07-01,2026-08-31,62,96.749,False
28,KTG-02,2026-09-01,2026-10-14,44,1119.553,True


# 8. Korelasi PM/polutan dengan cuaca
**Tujuan:** menguji apakah suhu, kelembapan, dan angin berhubungan dengan tiap polutan.
**Dua versi:** `semua_data` (tanpa pengecualian; pembanding) dan `tanpa_semua_anomali` (versi yang dipakai). Keduanya dihitung paralel.
**Cek otomatis:** n PM2.5 versi `tanpa_semua_anomali` harus sama dengan jumlah hari `normal` (21.691); bila tidak, notebook berhenti.
**Waktu:** bagian paling lama. Ubah `BOOT_B` ke 100–200 untuk uji cepat (hasil uji cepat jangan dipakai untuk laporan).

In [ ]:
# ==== 8. KORELASI CUACA (BOOTSTRAP) ====
BOOT_B = 1000      # 1000 = final (~10-15 menit); 100-200 = uji cepat

d_pm = d.copy()
for c in ["pm25", "pm10"]:
    d_pm.loc[~d_pm["station"].isin(PM_COMPARE), c] = np.nan
d_semua, d_tanpa = d_pm, d_pm[~mask].copy()

print(f"Bootstrap B={BOOT_B} untuk dua versi ...")
try:
    with ProcessPoolExecutor(max_workers=2) as ex:
        f1 = ex.submit(hitung_korelasi, "semua_data", d_semua, BOOT_B)
        f2 = ex.submit(hitung_korelasi, "tanpa_semua_anomali", d_tanpa, BOOT_B)
        K_raw = pd.concat([f1.result(), f2.result()], ignore_index=True)
except Exception as e:
    print("Paralel gagal (", e, ") -> jalan berurutan")
    K_raw = pd.concat([hitung_korelasi("semua_data", d_semua, BOOT_B),
                       hitung_korelasi("tanpa_semua_anomali", d_tanpa, BOOT_B)], ignore_index=True)
K = finalkan_korelasi(K_raw)

n_pm = int(K[(K.versi == "tanpa_semua_anomali") & (K.scope == "harian_gabungan") & (K.polutan == "pm25")]["n"].iloc[0])
n_normal = int((daily["status_data"] == "normal").sum())
print(f"Cek: n PM2.5 tanpa_semua_anomali = {n_pm} (harus {n_normal})")
assert n_pm == n_normal

print("\nKorelasi utama (dalam stasiun & bulan yang sama), versi tanpa_semua_anomali:")
display(K[(K.versi == "tanpa_semua_anomali") & (K.scope == "harian_bebas_musim")]
        [["polutan", "variabel", "rho", "ci95_lo", "ci95_hi", "n", "indikasi_eksploratif"]].round(3))

Bootstrap B=1000 untuk dua versi ...
  [tanpa_semua_anomali] harian_gabungan selesai (c_simultan=0.0357)
  [semua_data] harian_gabungan selesai (c_simultan=0.0350)
  [tanpa_semua_anomali] harian_dalam_stasiun selesai (c_simultan=0.0216)
  [semua_data] harian_dalam_stasiun selesai (c_simultan=0.0214)
  [tanpa_semua_anomali] harian_bebas_musim selesai (c_simultan=0.0224)
  [semua_data] harian_bebas_musim selesai (c_simultan=0.0221)
  [tanpa_semua_anomali] bulanan_dalam_stasiun selesai (c_simultan=0.1208)
  [semua_data] bulanan_dalam_stasiun selesai (c_simultan=0.1206)
Cek: n PM2.5 tanpa_semua_anomali = 21691 (harus 21691)

Korelasi utama (dalam stasiun & bulan yang sama), versi tanpa_semua_anomali:


,polutan,variabel,rho,ci95_lo,ci95_hi,n,indikasi_eksploratif
108,pm25,temp,-0.030,-0.045,-0.015,21691,tidak ada indikasi
109,pm25,rh,0.013,-0.001,0.026,21691,tidak ada indikasi
110,pm25,ws,-0.001,-0.010,0.009,21691,tidak ada indikasi
111,pm10,temp,-0.029,-0.045,-0.013,21691,tidak ada indikasi
112,pm10,rh,0.011,-0.004,0.027,21691,tidak ada indikasi
113,pm10,ws,-0.001,-0.012,0.010,21691,tidak ada indikasi
114,so2,temp,-0.014,-0.023,-0.004,23794,tidak ada indikasi
115,so2,rh,0.004,-0.009,0.018,23794,tidak ada indikasi
116,so2,ws,0.008,-0.004,0.019,23794,tidak ada indikasi
117,co,temp,-0.009,-0.023,0.004,23794,tidak ada indikasi


# 9. Ekspor ke Excel
## 9.1 Susun workbook
**Tujuan:** menulis semua tabel ke satu file Excel siap Looker.
**Fungsi `bangun_harian_v2`** menyusun sheet `harian_stasiun_v2` (data harian + status + `pm25_dipakai`; kolom `kategori_kawasan_sites` memuat kawasan sebenarnya untuk semua 27 stasiun).
**Sheet (10):** `catatan_perubahan`, `harian_stasiun_v2`, `weekday_weekend`, `episode_pm25`, `korelasi_cuaca`, `audit_dugaan_anomali`, `pola_jam_hari_stasiun`, `heatmap_jam_hari_v2`, `pola_jam_kawasan`, `audit_cleaning`.

In [ ]:
# ==== 9.1 FUNGSI EKSPOR + CATATAN ====
CATATAN_BARU = [
"weekday_weekend dihitung ulang dengan tanpa_semua_anomali (KTG >= 1 Sep 2026 + dugaan 8-14 Okt 2026) untuk SEMUA polutan, bukan hanya PM2.5, karena PM10 dan CO ikut terkontaminasi anomali. Metode: rata-rata bobot-stasiun-setara dari rata-rata stasiun per jenis hari.",
"weekday_weekend, gas kawasan pesisir: angka lama reproducible dan bukan akibat ppm belum dikonversi. Penyebabnya MER-01 yang hanya punya 2 hari gas valid tetapi berbobot 1/3 pada rata-rata bobot-stasiun-setara. Jangan dibaca sebagai efek akhir pekan.",
"episode_pm25 dihitung ulang dari PM2.5 setelah pengecualian (aturan >55,4 dan >= 3 hari berturut-turut). Rentang yang dikecualikan tetap ada sebagai baris terpisah dengan ada_nilai_belum_tervalidasi = True. Filter False untuk angka yang dipakai.",
"episode_pm25 kini memuat semua stasiun PM yang memenuhi aturan, bukan hanya stasiun gambut; kolom baru kategori_kawasan di ujung kanan. Filter kategori_kawasan = gambut/karhutla untuk set lama.",
"korelasi_cuaca: versi 'tanpa_KTG_anomali' diganti nama 'tanpa_semua_anomali' (KTG + dugaan 8-14 Okt, semua polutan). Versi 'semua_data' dipertahankan sebagai pembanding. Block bootstrap per bulan-kalender (seed 42) dihitung ulang untuk kedua versi.",
"korelasi_cuaca skop bulanan_dalam_stasiun: definisi aslinya tidak bisa direproduksi dari notebook yang ada. Dipakai definisi rekonstruksi (rata-rata bulanan per stasiun, minimal 16 hari valid, dikurangi rata-rata stasiun, Spearman). Perlakukan sebagai pembanding.",
"harian_stasiun_v2: kolom baru kategori_kawasan_sites (ujung kanan) berisi kawasan dari sites.csv untuk semua 27 stasiun. Jangan pakai kategori_kawasan lama untuk mengelompokkan gas (lima stasiun berisi 'tidak dianalisis (PM2.5)').",
]


CATATAN_V4 = [
"Workbook ini dibangun langsung dari dataset mentah (zip), tanpa workbook v3. Isinya hanya sheet yang bisa dihitung ulang dari dataset: harian_stasiun_v2, weekday_weekend, episode_pm25, korelasi_cuaca, audit_dugaan_anomali, pola_jam_hari_stasiun, heatmap_jam_hari_v2, pola_jam_kawasan, audit_cleaning. Sheet lain di v3 (kpi_ringkasan, peringkat_stasiun, faktor_lokasi_v2, korelasi_lokasi_v2, tren_bulanan_kawasan, musiman_kawasan_v2, korelasi_cuaca_viz, bandingan_kawasan, sektor_angin, ktg_44hari, lokasi_stasiun) tidak dihitung ulang di sini dan tetap berlaku dari v3.",
"status_data menandai hari-stasiun yang dikecualikan. Pakai pm25_dipakai (bukan pm25) di SEMUA grafik agar konsisten. Pengecualian: KTG-01/KTG-02 >= 1 Sep 2026 (88 hari-stasiun) dan dugaan anomali 8-14 Okt 2026 (48 hari-stasiun; stasiun BTM-01, JMB-01, KTB-01, PKN-01, PKU-01, STG-01 = dugaan ramp, PLB-01 = dugaan datar).",
] + CATATAN_BARU + [
"harian_stasiun_v2: provinsi, kota_kabupaten, dan site_type kini terisi untuk semua 27 stasiun (di v3 kosong untuk BJB-01, JMB-02, MDN-01, MER-01, PSW-01). Kolom kategori_kawasan lama dipertahankan agar kompatibel.",
]

def bangun_harian_v2(daily, raw, sites, PM_COMPARE, audit_dugaan):
    s = sites.copy(); s["station"] = s["location_id"].str.replace("LOC-", "", regex=False)
    jv = raw.groupby(["station", "date"])["pm25"].count().rename("jam_pm25_valid").reset_index()
    x = daily.merge(jv, on=["station", "date"], how="left")
    x = x.merge(s[["station", "provinsi", "kota_kabupaten", "site_type"]], on="station", how="left")
    x["pm_standar_ugm3"] = x["station"].isin(PM_COMPARE)
    x["kategori_kawasan_sites"] = x["kawasan"]
    x["kategori_kawasan"] = np.where(x["pm_standar_ugm3"], x["kawasan"], "tidak dianalisis (PM2.5)")
    x["anomali_ktg"] = x["station"].isin(["KTG-01", "KTG-02"]) & (x["date"] >= pd.Timestamp("2026-09-01"))
    jenis = audit_dugaan.set_index("station")["dugaan"]
    st = x["status_data"].replace({"PM2.5 tidak valid (jam valid < MIN_JAM_VALID)": "PM2.5 tidak valid (cakupan jam < 18)"})
    dug = x["status_data"].str.startswith("dugaan anomali")
    x["status_data"] = np.where(dug, x["station"].map(jenis) + " 8-14 Okt 2026 (perlu verifikasi)", st)
    x = x.rename(columns={"date": "tanggal"})
    cols = ["station", "tanggal", "pm25", "pm10", "so2", "co", "o3", "no2", "temp", "rh", "ws", "wd", "jam_pm25_valid", "provinsi",
            "kota_kabupaten", "kategori_kawasan", "site_type", "pm_standar_ugm3", "anomali_ktg", "status_data", "pm25_dipakai",
            "kategori_kawasan_sites"]
    x = x[cols].sort_values(["station", "tanggal"]).reset_index(drop=True)
    for c in ["pm25", "pm10", "so2", "co", "o3", "no2", "temp", "rh", "ws", "wd", "pm25_dipakai"]:
        x[c] = x[c].round(3)
    return x

def tulis_workbook(path, sheets, catatan):
    from openpyxl.styles import Font
    cat = pd.DataFrame({"no": range(1, len(catatan) + 1), "catatan": catatan})
    with pd.ExcelWriter(path, engine="openpyxl", datetime_format="yyyy-mm-dd") as xw:
        cat.to_excel(xw, sheet_name="catatan_perubahan", index=False)
        for nama, df in sheets.items():
            df.to_excel(xw, sheet_name=nama, index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "A2"
            for c in ws[1]: c.font = Font(bold=True)
        wc = xw.book["catatan_perubahan"]; wc.column_dimensions["B"].width = 140
    print("Tersimpan:", os.path.abspath(path))

## 9.2 Tulis file Excel
**Tujuan:** menyimpan workbook ke `/content`.
**Hasil:** `Anggota_2_data_v4_siap_looker.xlsx` (10 sheet). Pengunduhan dilakukan sekaligus di langkah 9.4 dalam satu file zip.

In [ ]:
# ==== 9.2 TULIS EXCEL ====
OUT_XLSX = "/content/Anggota_2_data_v4_siap_looker.xlsx"
harian = bangun_harian_v2(daily, raw, _sites, PM_COMPARE, audit_dugaan)
sheets = {"harian_stasiun_v2": harian, "weekday_weekend": ww, "episode_pm25": ep, "korelasi_cuaca": K,
          "audit_dugaan_anomali": audit_dugaan, "pola_jam_hari_stasiun": pola, "heatmap_jam_hari_v2": heatmap_v2,
          "pola_jam_kawasan": pjk, "audit_cleaning": audit_cleaning}
tulis_workbook(OUT_XLSX, sheets, CATATAN_V4)
print(f"{os.path.getsize(OUT_XLSX)/1e6:.1f} MB;", len(sheets) + 1, "sheet")

Tersimpan: /content/Anggota_2_data_v4_siap_looker.xlsx
3.2 MB; 10 sheet


## 9.3 Ringkasan temuan
**Tujuan:** mencetak angka kunci langsung dari hasil perhitungan (bukan diketik manual), sebagai bahan laporan.

In [ ]:
# ==== 9.3 RINGKASAN TEMUAN ====
g = ww[(ww.polutan == "PM2.5") & (ww.kategori_kawasan == "gambut/karhutla")].iloc[0]
print(f"1. Hari-stasiun PM2.5 dikecualikan : {int(daily['status_data'].str.startswith(('anomali','dugaan')).sum())} "
      f"(KTG {int(mask_ktg.sum())} baris; dugaan 8-14 Okt: {', '.join(DUGAAN_STASIUN)})")
print(f"2. PM2.5 gambut, akhir pekan vs hari kerja : {g.selisih_pct_weekend_vs_weekday*100:+.2f}% "
      f"(n hari kerja {int(g.n_hari_Weekday)})")
e = ep[(~ep.ada_nilai_belum_tervalidasi) & (ep.mulai >= "2026-01-01")].groupby("station_id").pm25_maks_harian.max().round(1)
print("3. Maks PM2.5 harian 2026 setelah pengecualian :", e.to_dict())
print(f"4. Episode PM2.5 tervalidasi : {int((~ep.ada_nilai_belum_tervalidasi).sum())} episode di "
      f"{ep[~ep.ada_nilai_belum_tervalidasi].station_id.nunique()} stasiun")
u = K[(K.versi == "tanpa_semua_anomali") & (K.indikasi_eksploratif.str.startswith("indikasi (")) & (K.scope == "harian_bebas_musim")]
print("5. Korelasi cuaca yang konsisten (CI simultan + tanda konsisten):",
      [f"{r.polutan}~{r.variabel} (rho={r.rho:+.2f})" for r in u.itertuples()])

1. Hari-stasiun PM2.5 dikecualikan : 136 (KTG 88 baris; dugaan 8-14 Okt: BTM-01, JMB-01, KTB-01, PKN-01, PKU-01, PLB-01, STG-01)
2. PM2.5 gambut, akhir pekan vs hari kerja : +0.28% (n hari kerja 4782)
3. Maks PM2.5 harian 2026 setelah pengecualian : {'BTM-01': 103.4, 'JKT-03': 60.1, 'JMB-01': 113.7, 'KTG-01': 96.7, 'KTG-02': 96.7, 'PKN-01': 148.7, 'PKU-01': 115.4, 'PLB-01': 192.9, 'SM2-01': 162.6, 'STG-01': 122.1}
4. Episode PM2.5 tervalidasi : 66 episode di 11 stasiun
5. Korelasi cuaca yang konsisten (CI simultan + tanda konsisten): ['o3~temp (rho=+0.16)', 'o3~rh (rho=-0.16)']


## 9.4 Bungkus semua hasil jadi satu zip dan unduh
**Tujuan:** memberi satu file `hasil_anggota2_v4.zip` yang langsung bisa diserahkan atau diunggah.
**Isi zip:**
- `Anggota_2_data_v4_siap_looker.xlsx` — workbook utama (10 sheet);
- `csv_audit/` — CSV pendukung dari proses pembersihan dan deteksi anomali (audit timestamp, log konversi ppm → µg/m³, flatline, zona waktu, dugaan anomali, dan sebagainya) sebagai bukti untuk laporan.
**Catatan:** nama zip memuat `hasil_anggota2` agar tidak terbaca sebagai zip dataset bila notebook dijalankan ulang.

In [ ]:
# ==== 9.4 ZIP HASIL + UNDUH ====
ZIP_OUT = "/content/hasil_anggota2_v4.zip"
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_XLSX, arcname=os.path.basename(OUT_XLSX))
    for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv"))):
        zf.write(f, arcname="csv_audit/" + os.path.basename(f))
with zipfile.ZipFile(ZIP_OUT) as zf:
    for i in zf.infolist():
        print(f"  {i.filename:<55} {i.file_size/1e6:6.2f} MB")
print(f"\nZip hasil: {ZIP_OUT} ({os.path.getsize(ZIP_OUT)/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(ZIP_OUT)
except ImportError:
    print("Bukan di Colab: ambil file dari path di atas.")

  Anggota_2_data_v4_siap_looker.xlsx                        3.23 MB
  csv_audit/audit_cleaning_per_stasiun.csv                  0.00 MB
  csv_audit/audit_dugaan_anomali_okt2026.csv                0.00 MB
  csv_audit/audit_flatline_runs.csv                         0.00 MB
  csv_audit/audit_pengukuran_multi_PLB01.csv                0.55 MB
  csv_audit/audit_timestamp.csv                             0.00 MB
  csv_audit/audit_timestamp_invalid.csv                     0.02 MB
  csv_audit/audit_zona_waktu.csv                            0.00 MB
  csv_audit/heatmap_jam_hari_v2.csv                         0.01 MB
  csv_audit/log_konversi_gas.csv                            0.00 MB
  csv_audit/pola_jam_hari_stasiun.csv                       0.26 MB
  csv_audit/pola_jam_kawasan.csv                            0.01 MB

Zip hasil: /content/hasil_anggota2_v4.zip (3.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# File Kesehatan
Perisapan File

In [ ]:
from google.colab import files
import os
os.makedirs("a1", exist_ok=True)

# A) Data mentah ke /content: HLT-01_health_outcomes_daily.csv dan hospitals.csv
files.upload()

# B) Paket integrasi Anggota 1 ke folder a1/ (dipisah agar dim_hospital.csv tidak tertimpa):
#    fact_health_daily_all27.csv, fact_health_air_daily.csv, fact_daily.csv,
#    bridge_station_hospital.csv, bridge_station_hospital_27.csv, dim_hospital.csv,
#    tabel_stasiun_analisis_pm.csv, fact_station_summary.csv,
#    correlation_health_lag_v7_A3final.py
os.chdir("a1"); files.upload(); os.chdir("..")

Pembersihan Data Mentah

In [ ]:
import pandas as pd
import numpy as np

IN_DIR = "."
OUT_DIR = "."

log = []
def add_log(file, masalah, n, keputusan):
    log.append({"file": file, "masalah": masalah, "baris_terdampak": int(n),
                "keputusan": keputusan, "pemilik": "Anggota 3"})

COUNT = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases",
         "copd_admissions", "resp", "cvd", "death"]

# ---------------- 1. HLT-01 ----------------
h = pd.read_csv(f"{IN_DIR}/HLT-01_health_outcomes_daily.csv")
raw_rows = len(h)
h.columns = h.columns.str.strip().str.lower()
h["hospital_id"] = h["hospital_id"].astype(str).str.strip().str.upper()
h["dow"] = h["dow"].astype(str).str.strip().str.title()
h["date"] = pd.to_datetime(h["date"], errors="coerce")

for c in COUNT + ["temp_c", "dptp_c", "rhum_pct", "time"]:
    h[c] = pd.to_numeric(h[c], errors="coerce")

add_log("HLT-01", "Missing value (semua kolom)", h.isna().any(axis=1).sum(), "Tidak ada, tidak perlu imputasi")
add_log("HLT-01", "Duplikat baris identik", h.duplicated().sum(), "Tidak ada")
add_log("HLT-01", "Duplikat kunci date+hospital_id", h.duplicated(["date", "hospital_id"]).sum(), "Tidak ada")
add_log("HLT-01", "Nilai kasus negatif / bukan bilangan bulat",
        ((h[COUNT] < 0) | ((h[COUNT] % 1) != 0)).any(axis=1).sum(), "Tidak ada")
add_log("HLT-01", "dow tidak sesuai date", (h["date"].dt.day_name() != h["dow"]).sum(), "Konsisten")
add_log("HLT-01", "Titik embun > suhu atau RH di luar 0-100",
        ((h["dptp_c"] > h["temp_c"]) | ~h["rhum_pct"].between(0, 100)).sum(), "Tidak ada")

# Kolom 'time' = urutan hari PER RUMAH SAKIT (5 RS mulai 2023-12-31, jadi bergeser 1 hari)
h = h.rename(columns={"time": "time_orig"})
h["day_idx"] = (h["date"] - pd.Timestamp("2023-12-31")).dt.days + 1
add_log("HLT-01", "Kolom time bukan indeks kalender global (per-RS, bergeser 1 hari untuk 5 RS)",
        (h["time_orig"] != h["day_idx"]).sum(),
        "time asli disimpan sbg time_orig; dibuat day_idx kalender global. Analisis memakai date/day_idx")

h["flag_resp_lt_components"] = h["resp"] < (h["resp_admissions_under5"] + h["resp_admissions_over60"])
add_log("HLT-01", "resp < under5 + over60 (resp bukan total penjumlahan komponen)",
        h["flag_resp_lt_components"].sum(),
        "Ditandai saja; resp diperlakukan sbg metrik terpisah. Jangan dijumlahkan dengan komponen")
h["flag_resp_zero"] = h["resp"] == 0
add_log("HLT-01", "resp = 0 pada hari tertentu", h["flag_resp_zero"].sum(), "Dipertahankan (hari tanpa kasus yang valid), ditandai")

h["flag_before_air_period"] = h["date"] < "2024-01-01"
add_log("HLT-01", "Tanggal 2023-12-31 (5 RS) di luar periode data udara (mulai 2024-01-01)",
        h["flag_before_air_period"].sum(), "Dipertahankan, ditandai; dikecualikan saat dijoin dengan PM2.5")

# Outlier: IQR x3 per RS (ditandai, tidak dibuang)
def out_mask(g, c):
    q1, q3 = g[c].quantile([.25, .75]); i = q3 - q1
    return (g[c] > q3 + 3 * i) | (g[c] < q1 - 3 * i)
for c in COUNT:
    h["out_" + c] = h.groupby("hospital_id", group_keys=False).apply(lambda g: out_mask(g, c))
h["is_outlier"] = h[[c for c in h if c.startswith("out_")]].any(axis=1)
h = h.drop(columns=[c for c in h if c.startswith("out_")])
add_log("HLT-01", "Outlier kasus (IQR x3 per RS)", h["is_outlier"].sum(),
        "Ditandai is_outlier, DIPERTAHANKAN: terkonsentrasi Agu-Okt & RS wilayah gambut/Sumatera-Kalimantan, diduga lonjakan asap")

exp_days = pd.date_range("2024-01-01", "2026-10-14")
miss = (h[h["date"] >= "2024-01-01"].groupby("hospital_id")["date"].nunique() != len(exp_days)).sum()
add_log("HLT-01", "RS dengan hari hilang (2024-01-01 s.d. 2026-10-14)", miss, "Tidak ada, data lengkap")

h["year"] = h["date"].dt.year
h["month"] = h["date"].dt.month
h = h.sort_values(["hospital_id", "date"]).reset_index(drop=True)
h["date"] = h["date"].dt.strftime("%Y-%m-%d")
add_log("HLT-01", "Jumlah baris awal vs akhir", raw_rows - len(h), "Tidak ada baris dihapus")

# ---------------- 2. hospitals.csv ----------------
hp = pd.read_csv(f"{IN_DIR}/hospitals.csv")
hp.columns = hp.columns.str.strip().str.lower()
for c in ["hospital_id", "nama_rumah_sakit", "kota_kabupaten", "kecamatan", "provinsi", "kelas_faskes"]:
    hp[c] = hp[c].astype(str).str.strip()
hp["hospital_id"] = hp["hospital_id"].str.upper()

prov_ref = {1: "DKI Jakarta", 2: "DKI Jakarta", 3: "DKI Jakarta", 4: "Kalimantan Tengah", 5: "Kalimantan Tengah",
 6: "Jawa Barat", 7: "Jawa Barat", 8: "DI Yogyakarta", 9: "Aceh", 10: "Sumatera Barat", 11: "Riau", 12: "Jambi",
 13: "Kepulauan Riau", 14: "Bengkulu", 15: "Papua Barat Daya", 16: "Jawa Timur", 17: "Sulawesi Selatan",
 18: "Sulawesi Tengah", 19: "Kalimantan Barat", 20: "Kalimantan Tengah", 21: "Sumatera Selatan", 22: "Lampung",
 23: "Papua Selatan", 24: "Sumatera Utara", 25: "Kalimantan Timur", 26: "Kalimantan Selatan", 27: "Jambi"}
ref = hp["hospital_id"].str[-2:].astype(int).map(prov_ref)
norm = lambda s: s.str.lower().str.replace(r"[^a-z]", "", regex=True)
beda = norm(hp["provinsi"]) != norm(ref)
add_log("hospitals.csv", "Provinsi RS tidak sesuai Daftar Stasiun casebook", beda.sum(),
        "Semua konsisten. Contoh 'Gambir=D.I. Yogyakarta' hanya salah ketik di casebook, bukan di file")
hp["provinsi_clean"] = ref
add_log("hospitals.csv", "Penulisan provinsi 'D.I. Yogyakarta' vs 'DI Yogyakarta' di sites/casebook",
        (hp["provinsi"] != hp["provinsi_clean"]).sum(), "Dibakukan di provinsi_clean agar bisa di-join dengan tabel master")
add_log("hospitals.csv", "kota_kabupaten tidak seragam (kadang tanpa awalan Kota/Kabupaten, mis. 'Jakarta Pusat')",
        hp["kota_kabupaten"].str.match(r"^(Kota|Kabupaten) ").eq(False).sum(), "Dibiarkan; join memakai hospital_id, bukan nama kota")
hp["kelas_faskes_grup"] = hp["kelas_faskes"].str.extract(r"Tipe ([ABC])")[0]
hp["is_rs_paru"] = hp["kelas_faskes"].str.contains("Paru", case=False)
add_log("hospitals.csv", "kelas_faskes memuat Tipe A & 'Tipe A (Khusus Paru)' padahal casebook menyebut B/C",
        hp["kelas_faskes"].str.contains("Tipe A").sum(),
        "Dipertahankan; dibuat kelas_faskes_grup & is_rs_paru. RS-XXX-06 (RS Paru) kemungkinan bias ke kasus pernapasan")
add_log("hospitals.csv", "kapasitas_tempat_tidur kosong", hp["kapasitas_tempat_tidur"].isna().sum(), "Tidak ada")
add_log("hospitals.csv", "hospital_id duplikat / jumlah RS", int(hp["hospital_id"].duplicated().sum()),
        f"{len(hp)} RS, id unik; semua id di HLT-01 ada di hospitals: {set(h.hospital_id) <= set(hp.hospital_id)}")

# ---------------- 3. Simpan ----------------
h.to_csv(f"{OUT_DIR}/clean_health_daily.csv", index=False)
hp.to_csv(f"{OUT_DIR}/dim_hospital.csv", index=False)
pd.DataFrame(log).to_csv(f"{OUT_DIR}/cleaning_log_health.csv", index=False)
print(h.shape, hp.shape)

Korelasi PM2.5 dan kasus

In [ ]:
%cd /content/a1
!python correlation_health_lag_v7_A3final.py
%cd /content

Tabel untuk Dashboard

In [ ]:
import pandas as pd, numpy as np

A1_DIR = "a1"         # paket Anggota 1
KOR_DIR = "a1"        # hasil Sel 4
OUT_DIR = "."

G = {"Pernapasan total": "resp", "ISPA balita": "resp_admissions_under5", "ISPA lansia": "resp_admissions_over60",
     "Asma": "asthma_emergency_cases", "PPOK": "copd_admissions", "Kardiovaskular": "cvd", "Kematian": "death"}

# ---------- Halaman 3 (Health Profile): 27 RS ----------
H = pd.read_csv(f"{A1_DIR}/fact_health_daily_all27.csv", parse_dates=["date"], low_memory=False)
P = H[H.health_record_status == "in_aqms_period"].copy()                 # 2024-01-01 s.d. 2026-10-14
P["bulan"] = P.date.dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d")
F = pd.read_csv(f"{A1_DIR}/fact_health_air_daily.csv", encoding="utf-8-sig", parse_dates=["date"])
P["nama_rs"] = (P.nama_rumah_sakit.str.replace("RSUD XXX ", "", regex=False)
                .str.replace("RS XXX ", "", regex=False).str.replace("XXX ", "", regex=False))
P["rs_paru"] = np.where(P.is_rs_paru, "Ya", "Tidak")
P["pasangan_pm25_layak"] = np.where(P.hospital_id.isin(set(F.hospital_id)), "Ya", "Tidak")

k = ["bulan", "hospital_id", "nama_rs", "pulau", "provinsi", "kelas_faskes", "kapasitas_tempat_tidur", "rs_paru", "pasangan_pm25_layak"]
base = P.groupby(k, as_index=False).agg(hari_rs=("date", "size"), **{n: (c, "sum") for n, c in G.items()})
long = base.melt(id_vars=k + ["hari_rs"], value_vars=list(G), var_name="kelompok", value_name="kasus")
long["kelompok"] = pd.Categorical(long["kelompok"], categories=list(G), ordered=True)      # urutan 7 kelompok tetap
long = long.sort_values(["hospital_id", "bulan", "kelompok"])
long["kelompok"] = long["kelompok"].astype(str)
long[["kapasitas_tempat_tidur", "kasus"]] = long[["kapasitas_tempat_tidur", "kasus"]].astype(int)
long[k + ["kelompok", "kasus", "hari_rs"]].to_csv(f"{OUT_DIR}/dash_health_monthly_long.csv", index=False)

(P.groupby(["bulan", "hospital_id", "nama_rs", "pulau", "provinsi", "rs_paru"], as_index=False)
  .agg(hari_rs=("date", "size"), hari_spike=("is_spike", "sum"), hari_outlier=("is_outlier", "sum"))
  .sort_values(["hospital_id", "bulan"]).to_csv(f"{OUT_DIR}/dash_flags_monthly.csv", index=False))

# ---------- Halaman 4 (Air-Health Relationship): 22 RS dengan pasangan PM2.5 layak ----------
F = F[F.pm25_lag0_status == "eligible_pm25_day"].copy()
F["dow"] = F.date.dt.dayofweek
for n, c in G.items():      # rasio kasus = kasus / rata-rata RS itu pada hari yang sama dalam minggu (1,0 = biasa)
    F["r_" + n] = F[c] / F.groupby(["hospital_id", "dow"])[c].transform("mean")
F["ktg"] = F.station_id.isin(["KTG-01", "KTG-02"]) & (F.date >= pd.Timestamp("2026-09-01"))
cuts = [0, 15, 35, 55, 100, 200, 1e9]
labs = ["1. <15", "2. 15–35", "3. 35–55", "4. 55–100", "5. 100–200", "6. >200"]
F["kelas"] = pd.cut(F.pm25_lag0, bins=cuts, labels=labs, right=False)

rows = []
for isl in ["Semua"] + sorted(F.pulau.unique()):
    S0 = F if isl == "Semua" else F[F.pulau == isl]
    for ep, S in [("Dengan KTG", S0), ("Tanpa KTG sejak 2026-09-01", S0[~S0.ktg])]:
        for n in G:
            for l in labs:
                m = S[S.kelas == l]
                rows.append(dict(pulau=isl, episode_KTG=ep, kelompok=n, kelas_pm25_ugm3=l,
                                 rasio_kasus=round(m["r_" + n].mean(), 3), hari_rs=len(m), n_rs=S.hospital_id.nunique(),
                                 data_sedikit="Ya" if len(m) < 30 else "Tidak"))      # < 30 hari-RS = tafsir hati-hati
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/dash_bins_pm25.csv", index=False)

F["bulan"] = F.date.dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d")
rows = []
for isl in ["Semua"] + sorted(F.pulau.unique()):
    S = F if isl == "Semua" else F[F.pulau == isl]
    g = S.groupby("bulan").agg(pm25_rata2=("pm25_lag0", "mean"), rasio_kasus_pernapasan=("r_Pernapasan total", "mean"),
                               n_rs=("hospital_id", "nunique")).reset_index()
    g.insert(0, "pulau", isl)
    rows.append(g)
pm = pd.concat(rows)
pm["pm25_rata2"] = pm.pm25_rata2.round(1)
pm["rasio_kasus_pernapasan"] = pm.rasio_kasus_pernapasan.round(3)
pm.to_csv(f"{OUT_DIR}/dash_pm_monthly.csv", index=False)

# ---------- Workbook 6 tab untuk halaman 4 (diunggah ke Google Sheets) ----------
lag = pd.read_csv(f"{KOR_DIR}/correlation_health_lag_0to7_eksplorasi.csv")
ci = pd.read_csv(f"{KOR_DIR}/correlation_health_lag.csv").drop(columns=["keterangan_versi", "ci_keterangan"])
isl_ = pd.read_csv(f"{KOR_DIR}/correlation_health_lag_by_island.csv").drop(columns=["keterangan_versi", "ci_keterangan"])
nama = P[["hospital_id", "nama_rs"]].drop_duplicates()           # nama RS ditambahkan agar mudah dibaca di Looker
rs = pd.read_csv(f"{KOR_DIR}/correlation_health_lag_by_hospital.csv").merge(nama, on="hospital_id", how="left")[
    ["nama_rs", "pulau", "hospital_id", "versi", "kelompok", "lag_hari", "r_pearson", "rho_spearman", "n_pasangan"]]
pmx = pd.read_csv(f"{OUT_DIR}/dash_pm_monthly.csv", parse_dates=["bulan"])
bins = pd.read_csv(f"{OUT_DIR}/dash_bins_pm25.csv")
with pd.ExcelWriter(f"{OUT_DIR}/A3_Data_Halaman4.xlsx", engine="openpyxl", datetime_format="yyyy-mm-dd") as w:
    for name, df in [("pm25_bulanan", pmx), ("pm25_kelas", bins), ("korelasi_lag", lag),
                     ("korelasi_ci", ci), ("korelasi_pulau", isl_), ("korelasi_rs", rs)]:
        df.to_excel(w, sheet_name=name, index=False)
print("selesai: 4 CSV dashboard + A3_Data_Halaman4.xlsx")